In [1]:
# ========================================
# CELLULE 1 : Imports et configuration
# ========================================
import pandas as pd
import pdfplumber
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliothèques chargées")
print(f"✅ Pandas version: {pd.__version__}")
print(f"✅ Environnement prêt pour l'extraction")

✅ Bibliothèques chargées
✅ Pandas version: 2.3.3
✅ Environnement prêt pour l'extraction


In [19]:
# ============================================================
# EXTRACTION Budget des Dépenses Courantes (1996-2023)
# Sources : Rapports BCC 2005, 2014, 2023
# Mémoire Master 2 IA - Analyse dépenses publiques RDC
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration des chemins
PDF_2005 = Path("../data/raw/rapport_annuel_2005.pdf")
PDF_2014 = Path("../data/raw/rapport_annuel_2014.pdf")
PDF_2023 = Path("../data/raw/rapport_annuel_2023.pdf")

# Pages correctes (index + 1)
PAGE_2005 = 152  # Index 151
PAGE_2014 = 117
PAGE_2023_153 = 153
PAGE_2023_154 = 154

# Dossier de sortie
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(exist_ok=True)

OUT_INTER_2005 = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2005.xlsx"
OUT_INTER_2014 = OUTPUT_DIR / "Budget_Depenses_Courantes_2005_2014.xlsx"
OUT_INTER_2023 = OUTPUT_DIR / "Budget_Depenses_Courantes_2014_2023.xlsx"
OUT_WIDE = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2023_WIDE.xlsx"
OUT_LONG = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2023_LONG.xlsx"

print("✅ Configuration chargée")

print("✅ Configuration chargée")
print(f"📂 Sortie : {OUTPUT_DIR}")

✅ Configuration chargée
✅ Configuration chargée
📂 Sortie : ..\data\processed


In [4]:
# ============================================================
# FONCTIONS UTILITAIRES
# ============================================================

# -------------------------
# 1. Nettoyage et conversion
# -------------------------
def to_number(val):
    if val is None:
        return np.nan
    v = str(val).strip().replace("'", "")
    if v in ["", "-", "–", "—", "nan", "NaN", "None"]:
        return np.nan
    v = v.replace("\u00A0", "").replace(" ", "").replace(",", ".")
    if not re.match(r"^-?\d+(\.\d+)?$", v):
        return np.nan
    try:
        return float(v)
    except:
        return np.nan

def clean_cells(df):
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all").reset_index(drop=True)
    df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
    return df

def drop_only_total_and_empty(df, col="Ministere"):
    keep = []
    for i, row in df.iterrows():
        name = str(row[col]).strip()
        if name == "" or name.lower() in ["nan", "none", "total"] or name.lower().startswith("source"):
            continue
        keep.append(i)
    return df.loc[keep].reset_index(drop=True)

def has_real_year_columns(df, years):
    ok_cols = sum(1 for y in years if y in df.columns and df[y].replace("", np.nan).notna().sum() >= 3)
    return ok_cols >= 5

# -------------------------
# 2. Normalisation ministères
# -------------------------
def normalize_basic(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().replace("\n", " ").replace("\t", " ")
    s = re.sub(r"\s+", " ", s)
    s_nfkd = unicodedata.normalize("NFKD", s)
    s = "".join([c for c in s_nfkd if not unicodedata.combining(c)])
    s = s.replace("'", "'").replace("–", "-").replace("—", "-")
    s = re.sub(r"\.+$", "", s).strip()
    return s.upper()

ALIAS_TO_CANON_SHORT = [
    (r"\bSANTE\b.*\bPUBLIQUE\b", "SANTE PUBLIQUE"), (r"\bMINISTERE\b.*\bSANTE\b", "SANTE PUBLIQUE"),
    (r"\bSANTE\b", "SANTE PUBLIQUE"), (r"\bEDUCATION\b.*\bNATIONALE\b", "EDUCATION NATIONALE"),
    (r"\bENSEIGNEMENT\b.*\bPRIMAIRE\b|\bEPST\b", "EDUCATION (EPST)"),
    (r"\bENSEIGNEMENT\b.*\bSUPERIEUR\b|\bESU\b", "ENSEIGNEMENT SUPERIEUR"),
    (r"\bFINANCES\b", "FINANCES"), (r"\bBUDGET\b", "BUDGET"), (r"\bTRESOR\b", "TRESOR"),
    (r"\bDEFENSE\b", "DEFENSE"), (r"\bINTERIEUR\b", "INTERIEUR"), (r"\bSECURITE\b", "SECURITE"),
    (r"\bPOLICE\b", "POLICE"), (r"\bJUSTICE\b", "JUSTICE"),
    (r"\bAFFAIRES\s+ETRANGER", "AFFAIRES ETRANGERES"), (r"\bAGRICULT", "AGRICULTURE"),
    (r"\bTRANSPORT", "TRANSPORT"), (r"\bINFRASTRUCT", "INFRASTRUCTURES"),
    (r"\bTRAVAUX\s+PUBLICS\b|\bTP\b", "TRAVAUX PUBLICS"), (r"\bENERGIE\b", "ENERGIE"),
    (r"\bELECTRIC", "ELECTRICITE"), (r"\bAFFAIRES\s+SOCIALES\b", "AFFAIRES SOCIALES"),
]

def canonicalize_ministere_short(raw_name: str) -> str:
    n = normalize_basic(raw_name)
    if n in {"", "NAN", "NONE"}:
        return ""
    for pattern, canon in ALIAS_TO_CANON_SHORT:
        if re.search(pattern, n, flags=re.IGNORECASE):
            return canon
    return n

FUNCTION_RULES = [
    (r"\bSANTE\b|HOPITAL|MEDEC", "SANTE"),
    (r"\bEDUCATION\b|ENSEIGNEMENT|UNIVERSIT|\bESU\b|\bEPST\b", "EDUCATION"),
    (r"\bFINANCES\b|\bBUDGET\b|\bTRESOR\b|IMPOT|DGI|DGRAD|DGDA", "FINANCES_PUBLIQUES"),
    (r"\bDEFENSE\b|ARME(E|ES)|MILITA", "DEFENSE"),
    (r"\bINTERIEUR\b|\bSECURITE\b|\bPOLICE\b", "SECURITE_INTERIEURE"),
    (r"\bJUSTICE\b|TRIBUNAL", "JUSTICE"), (r"\bAFFAIRES\s+ETRANGER", "DIPLOMATIE"),
    (r"\bAGRICULT", "AGRICULTURE"), (r"\bTRANSPORT\b|ROUTE|VOIRIE|\bINFRASTRUCT", "TRANSPORT_INFRA"),
    (r"\bENERGIE\b|ELECTRIC", "ENERGIE"),
]

def assign_function(ministere_canonique: str) -> str:
    n = normalize_basic(ministere_canonique)
    if n == "":
        return "AUTRE"
    for pattern, func in FUNCTION_RULES:
        if re.search(pattern, n, flags=re.IGNORECASE):
            return func
    return "AUTRE"

def enrich_ministere_columns(df, col="Ministere"):
    df = df.copy()
    df["Ministere_raw"] = df[col].astype(str)
    df["Ministere_canonique"] = df["Ministere_raw"].apply(canonicalize_ministere_short)
    df["Fonction"] = df["Ministere_canonique"].apply(assign_function)
    df["Ministere_key"] = df["Ministere_canonique"].apply(normalize_basic)
    df[col] = df["Ministere_canonique"]
    return df

# -------------------------
# 3. Extraction PDF
# -------------------------
def tabula_read(pdf_path, page, lattice=False, stream=False, area=None, guess=True):
    import tabula
    return tabula.read_pdf(pdf_path, pages=page, multiple_tables=True, lattice=lattice,
                          stream=stream, area=area, guess=guess, pandas_options={"header": None})

def pick_best_candidate(dfs, year_start=1996, year_end=2005):
    if not dfs:
        return None
    years = [str(y) for y in range(year_start, year_end + 1)]
    best_df, best_score = None, -1
    for d in dfs:
        df = clean_cells(d)
        first_rows = " ".join(df.head(5).astype(str).fillna("").values.flatten().tolist())
        year_hits = sum(1 for y in years if y in first_rows)
        score = df.shape[1] * 2 + year_hits * 5
        if score > best_score:
            best_score, best_df = score, df
    return best_df

def reshape_fixed_years(raw_df, year_start, year_end):
    years = [str(y) for y in range(year_start, year_end + 1)]
    df = clean_cells(raw_df)
    expected_cols = 1 + len(years)
    if df.shape[1] < expected_cols:
        raise RuntimeError("Pas assez de colonnes extraites.")
    df = df.iloc[:, :expected_cols].copy()
    df.columns = ["Ministere"] + years
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    for y in years:
        df[y] = df[y].apply(to_number)
    return df.reset_index(drop=True)

def drop_non_ministere_rows_2023(df, col="Ministere"):
    rubric_letter = re.compile(r"^\s*[A-Z]\.\s+")
    fiscal_keywords = [r"\bimpots?\b", r"\btaxe(s)?\b", r"\bdroits?\b", r"\baccises?\b",
                      r"\bvehicules?\b", r"\bimmatriculation\b", r"\bdouan(e|es)\b"]
    fiscal_regex = re.compile("|".join(fiscal_keywords), flags=re.IGNORECASE)
    
    def is_bad(name: str) -> bool:
        if name is None:
            return True
        raw = str(name).strip()
        if raw == "" or raw.lower() in ["nan", "none"]:
            return True
        if rubric_letter.search(raw):
            return True
        n = normalize_basic(raw)
        if fiscal_regex.search(n):
            return True
        return False
    
    keep = [i for i, row in df.iterrows() if not is_bad(row[col])]
    return df.loc[keep].reset_index(drop=True)

# -------------------------
# 4. Fusion
# -------------------------
def merge_two(left, right, prefer_right_years=None):
    years_left = sorted([c for c in left.columns if re.match(r"^(19|20)\d{2}$", str(c))], key=lambda x: int(x))
    years_right = sorted([c for c in right.columns if re.match(r"^(19|20)\d{2}$", str(c))], key=lambda x: int(x))
    
    base_left = ["Ministere", "Ministere_key"]
    base_right = ["Ministere", "Ministere_key"]
    if "Fonction" in left.columns: base_left.append("Fonction")
    if "Fonction" in right.columns: base_right.append("Fonction")
    if "Ministere_raw" in left.columns: base_left.append("Ministere_raw")
    if "Ministere_raw" in right.columns: base_right.append("Ministere_raw")
    
    m = pd.merge(left[base_left + years_left], right[base_right + years_right],
                on="Ministere_key", how="outer", suffixes=("_L", "_R"))
    
    m["Ministere_final"] = m.get("Ministere_R").where(
        m.get("Ministere_R").notna() & (m.get("Ministere_R").astype(str).str.strip() != ""),
        m.get("Ministere_L"))
    m.drop(columns=["Ministere_L", "Ministere_R"], inplace=True, errors="ignore")
    m.rename(columns={"Ministere_final": "Ministere"}, inplace=True)
    
    if "Fonction_L" in m.columns or "Fonction_R" in m.columns:
        m["Fonction"] = m.get("Fonction_R").where(
            m.get("Fonction_R").notna() & (m.get("Fonction_R").astype(str).str.strip() != ""),
            m.get("Fonction_L"))
        m.drop(columns=["Fonction_L", "Fonction_R"], inplace=True, errors="ignore")
    
    if prefer_right_years:
        for y in prefer_right_years:
            yl, yr = f"{y}_L", f"{y}_R"
            if yl in m.columns and yr in m.columns:
                m[y] = m[yr].combine_first(m[yl])
                m.drop(columns=[yl, yr], inplace=True)
    
    for y in years_left:
        col = f"{y}_L"
        if col in m.columns and y not in m.columns:
            m.rename(columns={col: y}, inplace=True)
    for y in years_right:
        col = f"{y}_R"
        if col in m.columns and y not in m.columns:
            m.rename(columns={col: y}, inplace=True)
    
    return m

print("✅ Toutes les fonctions chargées")

✅ Toutes les fonctions chargées


In [11]:
# ============================================================
# FONCTIONS D'EXTRACTION PAR RAPPORT (PDFPLUMBER)
# ============================================================

def extract_table_pdfplumber(pdf_path, page_num, table_settings=None):
    """Extrait un tableau d'une page PDF avec pdfplumber"""
    import pdfplumber
    
    if table_settings is None:
        table_settings = {
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
            "intersection_tolerance": 3,
        }
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num - 1]  # pdfplumber indexe à partir de 0
        tables = page.extract_tables(table_settings)
        
        if not tables:
            # Tentative avec stratégie alternative
            table_settings["vertical_strategy"] = "text"
            table_settings["horizontal_strategy"] = "text"
            tables = page.extract_tables(table_settings)
        
        return tables[0] if tables else None

def extract_2005(pdf_path, page):
    """Extrait le rapport 2005 (pages 152)"""
    print(f"   Extraction page {page}...")
    
    table_data = extract_table_pdfplumber(pdf_path, page)
    
    if table_data is None:
        raise RuntimeError(f"❌ Aucun tableau trouvé page {page}")
    
    # Convertir en DataFrame
    df = pd.DataFrame(table_data[1:], columns=table_data[0])
    df = clean_cells(df)
    
    # Identifier la colonne ministère (première colonne non vide)
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Ministere"})
    
    # Renommer les colonnes années (détecter automatiquement)
    years = [str(y) for y in range(1996, 2006)]
    year_cols = []
    
    for col in df.columns[1:]:
        col_str = str(col).strip()
        if col_str in years:
            year_cols.append(col_str)
        else:
            # Essayer de détecter l'année dans le contenu
            for year in years:
                if year in col_str:
                    year_cols.append(year)
                    break
    
    # Reconstruire avec colonnes propres
    expected_cols = ["Ministere"] + years
    df = df.iloc[:, :len(expected_cols)].copy()
    df.columns = expected_cols
    
    # Nettoyage
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    
    # Conversion numérique
    for y in years:
        df[y] = df[y].apply(to_number)
    
    return df.reset_index(drop=True)

def extract_2014(pdf_path, page):
    """Extrait le rapport 2014 (page 117)"""
    print(f"   Extraction page {page}...")
    
    table_data = extract_table_pdfplumber(pdf_path, page)
    
    if table_data is None:
        raise RuntimeError(f"❌ Aucun tableau trouvé page {page}")
    
    # Convertir en DataFrame
    df = pd.DataFrame(table_data[1:], columns=table_data[0])
    df = clean_cells(df)
    
    # Identifier colonnes
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Ministere"})
    
    years = [str(y) for y in range(2005, 2015)]
    expected_cols = ["Ministere"] + years
    df = df.iloc[:, :len(expected_cols)].copy()
    df.columns = expected_cols
    
    # Nettoyage
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    
    # Conversion numérique
    for y in years:
        df[y] = df[y].apply(to_number)
    
    return df.reset_index(drop=True)

def extract_2023_page(pdf_path, page_num, bbox=None):
    """Extrait une page du rapport 2023"""
    import pdfplumber
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num - 1]
        
        # Crop si bbox fourni (format: [x0, top, x1, bottom])
        if bbox:
            page = page.crop(bbox)
        
        table_data = page.extract_table()
        return table_data

def extract_2023(pdf_path):
    """Extrait le rapport 2023 (pages 153-154)"""
    print(f"   Extraction pages {PAGE_2023_153}-{PAGE_2023_154}...")
    
    # Extraire les deux pages
    # Convertir AREA [top, left, bottom, right] en bbox [x0, top, x1, bottom]
    # AREA_153 = [260, 20, 800, 580] → bbox = [20, 260, 580, 800]
    bbox_153 = (AREA_153[1], AREA_153[0], AREA_153[3], AREA_153[2])
    bbox_154 = (AREA_154[1], AREA_154[0], AREA_154[3], AREA_154[2])
    
    table_153 = extract_2023_page(pdf_path, PAGE_2023_153, bbox=bbox_153)
    table_154 = extract_2023_page(pdf_path, PAGE_2023_154, bbox=bbox_154)
    
    if table_153 is None or table_154 is None:
        raise RuntimeError("❌ Extraction 2023 échouée")
    
    # Combiner les deux pages
    all_rows = table_153 + table_154
    
    # Convertir en DataFrame
    df = pd.DataFrame(all_rows[1:], columns=all_rows[0])
    df = clean_cells(df)
    
    # Identifier colonnes
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Ministere"})
    
    years = [str(y) for y in range(2014, 2024)]
    expected_cols = ["Ministere"] + years
    df = df.iloc[:, :len(expected_cols)].copy()
    df.columns = expected_cols
    
    # Nettoyage
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    df = drop_non_ministere_rows_2023(df, "Ministere")
    
    # Conversion numérique
    for y in years:
        df[y] = df[y].apply(to_number)
    
    return df.reset_index(drop=True)

print("✅ Fonctions d'extraction (pdfplumber) chargées")

✅ Fonctions d'extraction (pdfplumber) chargées


In [15]:
# ============================================================
# FONCTIONS D'EXTRACTION PAR RAPPORT (PDFPLUMBER)
# ============================================================

def extract_table_pdfplumber(pdf_path, page_num, table_settings=None):
    """Extrait un tableau d'une page PDF avec pdfplumber"""
    import pdfplumber
    
    if table_settings is None:
        table_settings = {
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
            "intersection_tolerance": 3,
        }
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num - 1]  # pdfplumber indexe à partir de 0
        tables = page.extract_tables(table_settings)
        
        if not tables:
            # Tentative avec stratégie alternative
            table_settings["vertical_strategy"] = "text"
            table_settings["horizontal_strategy"] = "text"
            tables = page.extract_tables(table_settings)
        
        return tables[0] if tables else None

def extract_2005(pdf_path, page):
    """Extrait le rapport 2005 (pages 152)"""
    print(f"   Extraction page {page}...")
    
    table_data = extract_table_pdfplumber(pdf_path, page)
    
    if table_data is None:
        raise RuntimeError(f"❌ Aucun tableau trouvé page {page}")
    
    # Convertir en DataFrame
    df = pd.DataFrame(table_data[1:], columns=table_data[0])
    df = clean_cells(df)
    
    # Identifier la colonne ministère (première colonne non vide)
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Ministere"})
    
    # Renommer les colonnes années (détecter automatiquement)
    years = [str(y) for y in range(1996, 2006)]
    year_cols = []
    
    for col in df.columns[1:]:
        col_str = str(col).strip()
        if col_str in years:
            year_cols.append(col_str)
        else:
            # Essayer de détecter l'année dans le contenu
            for year in years:
                if year in col_str:
                    year_cols.append(year)
                    break
    
    # Reconstruire avec colonnes propres
    expected_cols = ["Ministere"] + years
    df = df.iloc[:, :len(expected_cols)].copy()
    df.columns = expected_cols
    
    # Nettoyage
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    
    # Conversion numérique
    for y in years:
        df[y] = df[y].apply(to_number)
    
    return df.reset_index(drop=True)

def extract_2014(pdf_path, page):
    """Extrait le rapport 2014 (page 117)"""
    print(f"   Extraction page {page}...")
    
    table_data = extract_table_pdfplumber(pdf_path, page)
    
    if table_data is None:
        raise RuntimeError(f"❌ Aucun tableau trouvé page {page}")
    
    # Convertir en DataFrame
    df = pd.DataFrame(table_data[1:], columns=table_data[0])
    df = clean_cells(df)
    
    # Identifier colonnes
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Ministere"})
    
    years = [str(y) for y in range(2005, 2015)]
    expected_cols = ["Ministere"] + years
    df = df.iloc[:, :len(expected_cols)].copy()
    df.columns = expected_cols
    
    # Nettoyage
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    
    # Conversion numérique
    for y in years:
        df[y] = df[y].apply(to_number)
    
    return df.reset_index(drop=True)

def extract_2023_page(pdf_path, page_num, bbox=None):
    """Extrait une page du rapport 2023"""
    import pdfplumber
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num - 1]
        
        # Crop si bbox fourni (format: [x0, top, x1, bottom])
        if bbox:
            page = page.crop(bbox)
        
        table_data = page.extract_table()
        return table_data

def extract_2023(pdf_path):
    """Extrait le rapport 2023 (pages 153-154)"""
    print(f"   Extraction pages {PAGE_2023_153}-{PAGE_2023_154}...")
    
    # Extraire les deux pages
    # Convertir AREA [top, left, bottom, right] en bbox [x0, top, x1, bottom]
    # AREA_153 = [260, 20, 800, 580] → bbox = [20, 260, 580, 800]
    bbox_153 = (AREA_153[1], AREA_153[0], AREA_153[3], AREA_153[2])
    bbox_154 = (AREA_154[1], AREA_154[0], AREA_154[3], AREA_154[2])
    
    table_153 = extract_2023_page(pdf_path, PAGE_2023_153, bbox=bbox_153)
    table_154 = extract_2023_page(pdf_path, PAGE_2023_154, bbox=bbox_154)
    
    if table_153 is None or table_154 is None:
        raise RuntimeError("❌ Extraction 2023 échouée")
    
    # Combiner les deux pages
    all_rows = table_153 + table_154
    
    # Convertir en DataFrame
    df = pd.DataFrame(all_rows[1:], columns=all_rows[0])
    df = clean_cells(df)
    
    # Identifier colonnes
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Ministere"})
    
    years = [str(y) for y in range(2014, 2024)]
    expected_cols = ["Ministere"] + years
    df = df.iloc[:, :len(expected_cols)].copy()
    df.columns = expected_cols
    
    # Nettoyage
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    df = drop_non_ministere_rows_2023(df, "Ministere")
    
    # Conversion numérique
    for y in years:
        df[y] = df[y].apply(to_number)
    
    return df.reset_index(drop=True)

print("✅ Fonctions d'extraction (pdfplumber) chargées")

✅ Fonctions d'extraction (pdfplumber) chargées


In [10]:
# ============================================================
# TEST D'EXTRACTION SIMPLE
# ============================================================

print("🔍 TEST D'EXTRACTION PDF")
print("="*60)

# Test 1 : Vérifier que les PDFs existent
print("\n1️⃣ Vérification des fichiers PDF :")
for nom, chemin in [("2005", PDF_2005), ("2014", PDF_2014), ("2023", PDF_2023)]:
    if chemin.exists():
        taille = chemin.stat().st_size / (1024*1024)  # en MB
        print(f"   ✅ {nom} : {chemin.name} ({taille:.1f} MB)")
    else:
        print(f"   ❌ {nom} : {chemin} - INTROUVABLE")

# Test 2 : Extraction simple d'une page
print("\n2️⃣ Test extraction page 152 (rapport 2005) :")
try:
    with pdfplumber.open(PDF_2005) as pdf:
        print(f"   📄 Nombre de pages : {len(pdf.pages)}")
        
        # Extraire la page 152
        if len(pdf.pages) >= 152:
            page = pdf.pages[151]  # Index 151 = page 152
            
            # Extraire le texte
            text = page.extract_text()
            print(f"   ✅ Texte extrait (premiers 300 caractères) :")
            print(f"   {text[:300]}")
            
            # Extraire le tableau
            print("\n   📊 Extraction du tableau...")
            table = page.extract_table()
            
            if table:
                print(f"   ✅ Tableau trouvé : {len(table)} lignes")
                print(f"   Aperçu des 3 premières lignes :")
                for i, row in enumerate(table[:3]):
                    print(f"      Ligne {i+1}: {row[:3]}...")  # 3 premières colonnes
            else:
                print(f"   ⚠️ Aucun tableau détecté automatiquement")
        else:
            print(f"   ❌ Le PDF n'a que {len(pdf.pages)} pages (page 152 n'existe pas)")
            
except Exception as e:
    print(f"   ❌ ERREUR : {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

🔍 TEST D'EXTRACTION PDF

1️⃣ Vérification des fichiers PDF :
   ✅ 2005 : rapport_annuel_2005.pdf (14.9 MB)
   ✅ 2014 : rapport_annuel_2014.pdf (5.3 MB)
   ✅ 2023 : rapport_annuel_2023.pdf (5.5 MB)

2️⃣ Test extraction page 152 (rapport 2005) :
   📄 Nombre de pages : 361
   ✅ Texte extrait (premiers 300 caractères) :
   Première partie EVOLUTION DE L’ACTIVITE ECONOMIQUE ET FINANCIERE, 2004-2005
132
Tableau II.20 Budget des dépenses courantes de l’Etat
répartition par ministère (en milliers de CDF)
1996 1997 1998 1999 2000 2001 2002 2003 2004 2005
I.Institutions politiques 18 800,7 15 656,9 21 457 28 913 775 128 2 6

   📊 Extraction du tableau...
   ✅ Tableau trouvé : 3 lignes
   Aperçu des 3 premières lignes :
      Ligne 1: ['', '1996', '1997']...
      Ligne 2: ['I.Institutions politiques\nPrésidence de la République.\nCompagnons de la Révolution.\nAssemblée Nationale & H.C.R.\nBureau du 1er ministre\nServ. techn. de la Prés.\nOrganismes auxiliaires(3)\nAutres(4)\nMagistrature, Cours 

In [20]:
# ============================================================
# EXTRACTION 2005 - GESTION DES CELLULES FUSIONNÉES
# ============================================================

def extract_2005_split_cells(pdf_path, page_num):
    """Extrait le tableau 2005 en séparant les cellules fusionnées"""
    print(f"   Extraction page {page_num}...")
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num - 1]
        
        # Extraire le tableau brut
        table = page.extract_table()
        
        if not table or len(table) < 2:
            raise RuntimeError("❌ Extraction échouée")
        
        # Le tableau a cette structure :
        # Ligne 0 : En-tête (années)
        # Ligne 1 : Tous les ministères fusionnés + valeurs fusionnées
        # Ligne 2 : TOTAL
        
        header = table[0]
        data_row = table[1]
        
        # Extraire les années de l'en-tête
        years = []
        for cell in header:
            if cell and str(cell).strip().isdigit():
                year = str(cell).strip()
                if 1996 <= int(year) <= 2005:
                    years.append(year)
        
        print(f"   ✅ Années détectées : {years}")
        
        # Séparer les ministères (colonne 0, ligne 1)
        ministeres_text = data_row[0]
        ministeres_lines = [line.strip() for line in ministeres_text.split('\n') if line.strip()]
        
        # Séparer les valeurs pour chaque année
        values_by_year = {}
        for i, year in enumerate(years):
            col_idx = i + 1  # +1 car colonne 0 = ministères
            if col_idx < len(data_row):
                values_text = data_row[col_idx]
                values = [v.strip() for v in values_text.split('\n') if v.strip()]
                values_by_year[year] = values
        
        # Vérifier cohérence
        nb_ministeres = len(ministeres_lines)
        print(f"   ✅ Ministères détectés : {nb_ministeres}")
        
        for year, vals in values_by_year.items():
            if len(vals) != nb_ministeres:
                print(f"   ⚠️ {year} : {len(vals)} valeurs vs {nb_ministeres} ministères")
        
        # Construire le DataFrame
        data = []
        for i, ministere in enumerate(ministeres_lines):
            row = {'Ministere': ministere}
            for year in years:
                if year in values_by_year and i < len(values_by_year[year]):
                    row[year] = values_by_year[year][i]
                else:
                    row[year] = np.nan
            data.append(row)
        
        df = pd.DataFrame(data)
        
        # Conversion numérique
        for year in years:
            df[year] = df[year].apply(to_number)
        
        return df.reset_index(drop=True)

# Test d'extraction
print("🔎 Test extraction 2005...")
print("-" * 60)

try:
    df2005 = extract_2005_split_cells(PDF_2005, PAGE_2005)
    df2005 = enrich_ministere_columns(df2005, col="Ministere")
    
    print(f"✅ Extraction réussie : {df2005.shape[0]} lignes x {df2005.shape[1]} colonnes")
    print("\n📊 Aperçu des données :")
    print(df2005[['Ministere', 'Fonction', '1996', '1997', '2005']].head(10))
    
    # Sauvegarder
    cols_2005 = ["Ministere", "Fonction"] + [str(y) for y in range(1996, 2006)]
    df2005[cols_2005].to_excel(OUT_INTER_2005, index=False, engine='openpyxl')
    print(f"\n✅ Fichier sauvegardé : {OUT_INTER_2005.name}")
    
except Exception as e:
    print(f"❌ ERREUR : {e}")
    import traceback
    traceback.print_exc()

🔎 Test extraction 2005...
------------------------------------------------------------
   Extraction page 152...
   ✅ Années détectées : ['1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005']
   ✅ Ministères détectés : 64
   ⚠️ 1996 : 59 valeurs vs 64 ministères
   ⚠️ 1997 : 59 valeurs vs 64 ministères
   ⚠️ 1998 : 59 valeurs vs 64 ministères
   ⚠️ 1999 : 59 valeurs vs 64 ministères
   ⚠️ 2000 : 59 valeurs vs 64 ministères
   ⚠️ 2001 : 60 valeurs vs 64 ministères
   ⚠️ 2002 : 61 valeurs vs 64 ministères
   ⚠️ 2003 : 62 valeurs vs 64 ministères
   ⚠️ 2004 : 62 valeurs vs 64 ministères
   ⚠️ 2005 : 63 valeurs vs 64 ministères
✅ Extraction réussie : 64 lignes x 15 colonnes

📊 Aperçu des données :
                         Ministere Fonction      1996      1997        2005
0        I.INSTITUTIONS POLITIQUES    AUTRE  18800.70  15656.90  22053066.0
1      PRESIDENCE DE LA REPUBLIQUE    AUTRE   4615.10  10816.90   9762441.0
2      COMPAGNONS DE LA REVOLUTION    AUTRE

In [21]:
# ============================================================
# VÉRIFICATION DU FICHIER 2005 GÉNÉRÉ
# ============================================================

print("📂 VÉRIFICATION DU FICHIER GÉNÉRÉ")
print("="*60)

# Charger le fichier sauvegardé
df_check = pd.read_excel(OUT_INTER_2005)

print(f"✅ Fichier chargé : {OUT_INTER_2005.name}")
print(f"   Dimensions : {df_check.shape[0]} lignes x {df_check.shape[1]} colonnes")
print(f"   Colonnes : {list(df_check.columns)}")

print("\n📊 Statistiques des valeurs manquantes par année :")
years_2005 = [str(y) for y in range(1996, 2006)]
for year in years_2005:
    missing = df_check[year].isna().sum()
    pct = (missing / len(df_check)) * 100
    print(f"   {year} : {missing}/{len(df_check)} manquantes ({pct:.1f}%)")

print("\n📋 Aperçu complet (15 premières lignes) :")
print(df_check.head(15).to_string())

print("\n💾 Fichier disponible dans :")
print(f"   {OUT_INTER_2005.absolute()}")

print("\n✅ Le fichier est prêt pour vérification manuelle !")
print("   Ouvrez-le dans Excel pour valider les données extraites.")

📂 VÉRIFICATION DU FICHIER GÉNÉRÉ
✅ Fichier chargé : Budget_Depenses_Courantes_1996_2005.xlsx
   Dimensions : 64 lignes x 12 colonnes
   Colonnes : ['Ministere', 'Fonction', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005']

📊 Statistiques des valeurs manquantes par année :
   1996 : 13/64 manquantes (20.3%)
   1997 : 14/64 manquantes (21.9%)
   1998 : 11/64 manquantes (17.2%)
   1999 : 28/64 manquantes (43.8%)
   2000 : 19/64 manquantes (29.7%)
   2001 : 12/64 manquantes (18.8%)
   2002 : 9/64 manquantes (14.1%)
   2003 : 10/64 manquantes (15.6%)
   2004 : 11/64 manquantes (17.2%)
   2005 : 10/64 manquantes (15.6%)

📋 Aperçu complet (15 premières lignes) :
                          Ministere    Fonction      1996      1997     1998      1999      2000       2001        2002        2003        2004        2005
0         I.INSTITUTIONS POLITIQUES       AUTRE  18800.70  15656.90  21457.0   28913.0  775128.0  2675842.0   8695683.0  26454601.0  16741139.0  2205

In [22]:
# ============================================================
# EXTRACTION DES RAPPORTS 2014 ET 2023
# ============================================================

def extract_similar_structure(pdf_path, page_num, year_start, year_end):
    """Extrait un tableau avec la même structure (cellules fusionnées)"""
    print(f"   Extraction page {page_num}...")
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num - 1]
        table = page.extract_table()
        
        if not table or len(table) < 2:
            raise RuntimeError("❌ Extraction échouée")
        
        header = table[0]
        data_row = table[1]
        
        # Extraire les années
        years = []
        for cell in header:
            if cell and str(cell).strip().isdigit():
                year = str(cell).strip()
                if year_start <= int(year) <= year_end:
                    years.append(year)
        
        print(f"   ✅ Années détectées : {years}")
        
        # Séparer les ministères
        ministeres_text = data_row[0]
        ministeres_lines = [line.strip() for line in ministeres_text.split('\n') if line.strip()]
        
        # Séparer les valeurs
        values_by_year = {}
        for i, year in enumerate(years):
            col_idx = i + 1
            if col_idx < len(data_row):
                values_text = data_row[col_idx]
                values = [v.strip() for v in values_text.split('\n') if v.strip()]
                values_by_year[year] = values
        
        nb_ministeres = len(ministeres_lines)
        print(f"   ✅ Ministères détectés : {nb_ministeres}")
        
        for year, vals in values_by_year.items():
            if len(vals) != nb_ministeres:
                print(f"   ⚠️ {year} : {len(vals)} valeurs vs {nb_ministeres} ministères")
        
        # Construire le DataFrame
        data = []
        for i, ministere in enumerate(ministeres_lines):
            row = {'Ministere': ministere}
            for year in years:
                if year in values_by_year and i < len(values_by_year[year]):
                    row[year] = values_by_year[year][i]
                else:
                    row[year] = np.nan
            data.append(row)
        
        df = pd.DataFrame(data)
        
        # Conversion numérique
        for year in years:
            df[year] = df[year].apply(to_number)
        
        return df.reset_index(drop=True)

print("🔎 EXTRACTION DES 3 RAPPORTS BCC")
print("="*60)

# Rapport 2014
print("\n📄 2/3 Rapport 2014 (2005-2014)...")
try:
    df2014 = extract_similar_structure(PDF_2014, PAGE_2014, 2005, 2014)
    df2014 = enrich_ministere_columns(df2014, col="Ministere")
    
    print(f"   ✅ Extraction réussie : {df2014.shape}")
    
    cols_2014 = ["Ministere", "Fonction"] + [str(y) for y in range(2005, 2015)]
    df2014[cols_2014].to_excel(OUT_INTER_2014, index=False, engine='openpyxl')
    print(f"   ✅ Sauvegardé : {OUT_INTER_2014.name}")
    
except Exception as e:
    print(f"   ❌ ERREUR : {e}")
    import traceback
    traceback.print_exc()

# Rapport 2023 (pages 153-154)
print("\n📄 3/3 Rapport 2023 (2014-2023)...")
try:
    # Extraire les 2 pages et combiner
    all_ministeres = []
    all_values_by_year = {}
    
    for page_num in [PAGE_2023_153, PAGE_2023_154]:
        print(f"   Extraction page {page_num}...")
        
        with pdfplumber.open(PDF_2023) as pdf:
            page = pdf.pages[page_num - 1]
            table = page.extract_table()
            
            if table and len(table) >= 2:
                # Page 153 a l'en-tête, page 154 continue les données
                if page_num == PAGE_2023_153:
                    header = table[0]
                    years = []
                    for cell in header:
                        if cell and str(cell).strip().isdigit():
                            year = str(cell).strip()
                            if 2014 <= int(year) <= 2023:
                                years.append(year)
                    
                    # Initialiser dictionnaire
                    for year in years:
                        all_values_by_year[year] = []
                
                # Extraire les données (ligne 1+)
                data_row = table[1] if len(table) > 1 else None
                if data_row:
                    ministeres_text = data_row[0]
                    ministeres = [line.strip() for line in ministeres_text.split('\n') if line.strip()]
                    all_ministeres.extend(ministeres)
                    
                    for i, year in enumerate(years):
                        col_idx = i + 1
                        if col_idx < len(data_row):
                            values_text = data_row[col_idx]
                            values = [v.strip() for v in values_text.split('\n') if v.strip()]
                            all_values_by_year[year].extend(values)
    
    print(f"   ✅ Années : {years}")
    print(f"   ✅ Total ministères : {len(all_ministeres)}")
    
    # Construire DataFrame
    data = []
    for i, ministere in enumerate(all_ministeres):
        # Filtrer lignes parasites (impôts, taxes)
        if re.match(r'^\s*[A-Z]\.\s+', ministere):
            continue
        
        row = {'Ministere': ministere}
        for year in years:
            if i < len(all_values_by_year[year]):
                row[year] = all_values_by_year[year][i]
            else:
                row[year] = np.nan
        data.append(row)
    
    df2023 = pd.DataFrame(data)
    
    # Conversion numérique
    for year in years:
        df2023[year] = df2023[year].apply(to_number)
    
    df2023 = enrich_ministere_columns(df2023, col="Ministere")
    
    print(f"   ✅ Extraction réussie : {df2023.shape}")
    
    cols_2023 = ["Ministere", "Fonction"] + [str(y) for y in range(2014, 2024)]
    df2023[cols_2023].to_excel(OUT_INTER_2023, index=False, engine='openpyxl')
    print(f"   ✅ Sauvegardé : {OUT_INTER_2023.name}")
    
except Exception as e:
    print(f"   ❌ ERREUR : {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)
print("✅ EXTRACTION TERMINÉE !")
print("="*60)

🔎 EXTRACTION DES 3 RAPPORTS BCC

📄 2/3 Rapport 2014 (2005-2014)...
   Extraction page 117...
   ✅ Années détectées : ['2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014']
   ✅ Ministères détectés : 60
   ⚠️ 2005 : 59 valeurs vs 60 ministères
   ⚠️ 2014 : 58 valeurs vs 60 ministères
   ✅ Extraction réussie : (60, 15)
   ✅ Sauvegardé : Budget_Depenses_Courantes_2005_2014.xlsx

📄 3/3 Rapport 2023 (2014-2023)...
   Extraction page 153...
   Extraction page 154...
   ✅ Années : []
   ✅ Total ministères : 2
   ✅ Extraction réussie : (2, 5)
   ❌ ERREUR : "['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'] not in index"

✅ EXTRACTION TERMINÉE !


Traceback (most recent call last):
  File "C:\Users\LEGION\AppData\Local\Temp\ipykernel_20632\2608296204.py", line 160, in <module>
    df2023[cols_2023].to_excel(OUT_INTER_2023, index=False, engine='openpyxl')
  File "c:\Users\LEGION\Music\Memoire Master\venv\lib\site-packages\pandas\core\frame.py", line 4119, in __getitem__
    indexer = self.columns._get_indexer_strict(key, "columns")[1]
  File "c:\Users\LEGION\Music\Memoire Master\venv\lib\site-packages\pandas\core\indexes\base.py", line 6212, in _get_indexer_strict
    self._raise_if_missing(keyarr, indexer, axis_name)
  File "c:\Users\LEGION\Music\Memoire Master\venv\lib\site-packages\pandas\core\indexes\base.py", line 6264, in _raise_if_missing
    raise KeyError(f"{not_found} not in index")
KeyError: "['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'] not in index"


In [29]:
# ============================================================
# EXTRACTION RAPPORT 2023 - VERSION FINALE
# ============================================================

print("📄 Extraction Rapport 2023 (2014-2023)")
print("="*60)

# Étape 1 : Extraire les années depuis le TEXTE de la page
with pdfplumber.open(PDF_2023) as pdf:
    page = pdf.pages[152]  # Page 153
    text = page.extract_text()
    
    # Chercher la ligne contenant les années
    lines = text.split('\n')
    years = []
    
    for line in lines[:30]:  # Chercher dans les 30 premières lignes
        # Ligne avec années : "2014 2015 2016 2017 2018 2019 2020 2021 2022 2023"
        year_candidates = [str(y) for y in range(2014, 2024)]
        found_in_line = [y for y in year_candidates if y in line]
        
        # Si on trouve au moins 5 années consécutives dans la ligne
        if len(found_in_line) >= 5:
            years = found_in_line
            print(f"✅ Années trouvées dans le texte : {years}")
            break
    
    if not years:
        # Plan B : utiliser toutes les années 2014-2023
        years = [str(y) for y in range(2014, 2024)]
        print(f"⚠️ Années non détectées, utilisation par défaut : {years}")

# Étape 2 : Extraire les tableaux des pages 153-154
all_rows = []

for page_num in [153, 154]:
    with pdfplumber.open(PDF_2023) as pdf:
        page = pdf.pages[page_num - 1]
        table = page.extract_table()
        
        if table:
            print(f"   Page {page_num} : {len(table)} lignes extraites")
            all_rows.extend(table)

print(f"   Total lignes extraites : {len(all_rows)}")

# Étape 3 : Construire le DataFrame
data = []

for row in all_rows:
    if not row or len(row) == 0:
        continue
    
    ministere = str(row[0]).strip() if row[0] else ""
    
    # Filtrer lignes inutiles
    if not ministere or ministere.lower() in ['', 'total', 'source']:
        continue
    if 'tableau' in ministere.lower():
        continue
    
    # Construire la ligne de données
    row_data = {'Ministere': ministere}
    
    # Les 10 colonnes suivantes = les 10 années (2014-2023)
    for i, year in enumerate(years[:10]):  # Max 10 années
        col_idx = i + 1
        if col_idx < len(row):
            row_data[year] = row[col_idx]
        else:
            row_data[year] = np.nan
    
    data.append(row_data)

df2023 = pd.DataFrame(data)

print(f"\n✅ DataFrame créé : {df2023.shape}")

# Étape 4 : Conversion numérique
for year in years:
    if year in df2023.columns:
        df2023[year] = df2023[year].apply(to_number)

# Étape 5 : Enrichissement
df2023 = enrich_ministere_columns(df2023, col="Ministere")

print(f"   Ministères : {df2023.shape[0]}")
print(f"   Années : {len(years)} ({years[0]}-{years[-1]})")

# Aperçu
print("\n📊 Aperçu des 10 premières lignes :")
display_cols = ['Ministere', 'Fonction'] + years[:3] + [years[-1]]
print(df2023[display_cols].head(10).to_string())

# Statistiques valeurs manquantes
print(f"\n📊 Valeurs manquantes par année :")
for year in years:
    if year in df2023.columns:
        missing = df2023[year].isna().sum()
        pct = (missing / len(df2023)) * 100
        print(f"   {year} : {missing}/{len(df2023)} ({pct:.1f}%)")

# Sauvegarder
cols_2023 = ["Ministere", "Fonction"] + years
df2023[cols_2023].to_excel(OUT_INTER_2023, index=False, engine='openpyxl')
print(f"\n✅ Fichier sauvegardé : {OUT_INTER_2023.name}")

print("\n" + "="*60)
print("✅ EXTRACTION DES 3 RAPPORTS TERMINÉE !")
print("="*60)

📄 Extraction Rapport 2023 (2014-2023)
✅ Années trouvées dans le texte : ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']
   Page 153 : 33 lignes extraites
   Page 154 : 28 lignes extraites
   Total lignes extraites : 61

✅ DataFrame créé : (60, 11)
   Ministères : 60
   Années : 10 (2014-2023)

📊 Aperçu des 10 premières lignes :
                         Ministere Fonction      2014      2015      2016        2023
0        I.INSTITUTIONS POLITIQUES    AUTRE  345889.1  341929.2  205727.4  1259897.40
1      PRESIDENCE DE LA REPUBLIQUE    AUTRE   63191.5   65694.7   43028.5   235948.20
2      ASSEMBLEE NATIONALE & SENAT    AUTRE  142809.9  148161.3   91711.0   708971.85
3                     PRIMATURE(1)    AUTRE   19140.6   19814.9   12078.9    84277.71
4        ORGANISMES AUXILIAIRES(2)    AUTRE   15656.9   15822.1       NaN     1756.83
5                        AUTRES(3)    AUTRE    3273.6   15822.1   12947.7    92787.60
6  MAGISTRATURE, COURS & TRIBUNAUX 

In [30]:
# ============================================================
# EXTRACTION RIGOUREUSE 2005 - PAGE 152
# ============================================================

print("📄 EXTRACTION RIGOUREUSE - RAPPORT 2005 PAGE 152")
print("="*60)

# Méthode 1 : Utiliser les coordonnées exactes du tableau
# Vous allez définir manuellement la zone du tableau

with pdfplumber.open(PDF_2005) as pdf:
    page = pdf.pages[151]  # Page 152
    
    # Définir la zone EXACTE du tableau (à ajuster selon votre PDF)
    # Format : (x0, top, x1, bottom) en points
    # Vous pouvez utiliser page.width et page.height comme référence
    
    print(f"Dimensions de la page : {page.width} x {page.height} points")
    
    # Zone du tableau (À AJUSTER - exemple)
    # Ces coordonnées sont à affiner selon votre PDF
    bbox = (
        40,    # x0 (marge gauche)
        180,   # top (début du tableau)
        page.width - 40,  # x1 (marge droite)
        page.height - 100  # bottom (fin du tableau)
    )
    
    # Crop la page à la zone du tableau
    cropped = page.crop(bbox)
    
    # Paramètres d'extraction stricts
    table_settings = {
        "vertical_strategy": "lines",
        "horizontal_strategy": "lines",
        "snap_tolerance": 5,
        "join_tolerance": 5,
        "edge_min_length": 10,
        "min_words_vertical": 3,
        "intersection_tolerance": 5,
    }
    
    # Extraire le tableau
    table = cropped.extract_table(table_settings)
    
    if not table:
        print("❌ Échec avec 'lines', essai avec 'text'...")
        table_settings["vertical_strategy"] = "text"
        table_settings["horizontal_strategy"] = "text"
        table = cropped.extract_table(table_settings)
    
    if table:
        print(f"✅ Tableau extrait : {len(table)} lignes x {len(table[0])} colonnes")
        
        # Afficher les 10 premières lignes
        print("\n📋 Structure du tableau extrait :")
        for i, row in enumerate(table[:10]):
            print(f"\nLigne {i}:")
            for j, cell in enumerate(row[:12]):  # 12 premières colonnes
                content = str(cell)[:50] if cell else "VIDE"
                print(f"   Col {j}: {content}")
    else:
        print("❌ Extraction échouée même avec crop")

print("\n" + "="*60)

📄 EXTRACTION RIGOUREUSE - RAPPORT 2005 PAGE 152
Dimensions de la page : 600.94489 x 853.22833 points
✅ Tableau extrait : 2 lignes x 12 colonnes

📋 Structure du tableau extrait :

Ligne 0:
   Col 0: Assemblée Nationale & H.C.R. 13 301,9 2 219,6 2 78
   Col 1: Assemblée Nationale & H.C.R.
Bureau du 1er ministr
   Col 2: 13 301,9
883,7
-
-
-
-
6 145,77
147,5
184,4
79,1
1
   Col 3: 2 219,6
2 620,5
-
-
-
-
15 706,99
917,0
1 245,1
22
   Col 4: 2 789
3 218
-
-
-
2 146
91 341
4 480
5 307
913
18 
   Col 5: -
-
7182
-
-
6 068
110 643
3 495
3 626
-
38 041
-

   Col 6: 50 000
-
-
-
-
64 975
956 073
82 150
5 940
2 479
4
   Col 7: -
-
-
315 223
-
184 306
21 000
40 000
5 877 060
84
   Col 8: 2 483 428
433 581
-
1 006 931
-
768 096
96 706
75 
   Col 9: 1 474 365
677 853
-
4 795 095
310 743
829 972
-
-

   Col 10: 3 576 198
693 978
-
-
1 869 819
838 429
-
-
31 988
   Col 11: 5 003 106
693 978
-
2 090 923
3 664 189
838 429
-


Ligne 1:
   Col 0: VIDE
   Col 1: TOTAL
   Col 2: 90 958,7
   Col 3: 699 314,

In [4]:
# ============================================================
# EXTRACTION RAPPORT 2005 - LATTICE MODE
# ============================================================

import tabula
import pandas as pd
import numpy as np

print("📄 EXTRACTION RAPPORT 2005 - LATTICE MODE")
print("="*60)

# Essayer lattice=True (détection des lignes du tableau)
dfs = tabula.read_pdf(
    str(PDF_2005),
    pages=152,
    lattice=True,           # Détection des bordures
    multiple_tables=True,
    pandas_options={"header": None}
)

print(f"✅ {len(dfs)} tableau(x) extrait(s)\n")

# Tester chaque tableau
for i, df in enumerate(dfs):
    print(f"📊 Tableau {i+1} : {df.shape[0]} lignes x {df.shape[1]} colonnes")
    
    # Chercher celui avec ~11 colonnes
    if df.shape[1] >= 10 and df.shape[0] > 50:
        print(f"   ⭐ CANDIDAT PRINCIPAL")
        print(f"\n   Ligne 0 (en-tête potentiel) :")
        print(f"   {list(df.iloc[0].values[:11])}")
        print(f"\n   Ligne 1 (données) :")
        print(f"   {df.iloc[1, 0]} | {list(df.iloc[1, 1:6].values)}")
        
        # Afficher aperçu complet
        print(f"\n   Aperçu des 5 premières lignes :")
        print(df.head(5).to_string())
        print()
        
        # Si c'est le bon, le sauvegarder
        if df.shape[1] == 11:
            df_2005_raw = df.copy()
            print(f"   ✅ Tableau sauvegardé dans 'df_2005_raw'")

print("="*60)

📄 EXTRACTION RAPPORT 2005 - LATTICE MODE
✅ 2 tableau(x) extrait(s)

📊 Tableau 1 : 4 lignes x 11 colonnes
📊 Tableau 2 : 4 lignes x 11 colonnes


In [5]:
# ============================================================
# ANALYSE DES 2 TABLEAUX DÉTECTÉS
# ============================================================

import tabula

print("📄 ANALYSE DES 2 TABLEAUX")
print("="*60)

dfs = tabula.read_pdf(
    str(PDF_2005),
    pages=152,
    lattice=True,
    multiple_tables=True,
    pandas_options={"header": None}
)

for i, df in enumerate(dfs):
    print(f"\n{'='*60}")
    print(f"📊 TABLEAU {i+1} - {df.shape[0]} lignes x {df.shape[1]} colonnes")
    print('='*60)
    
    # Afficher TOUTES les lignes
    print("\nContenu complet :")
    for row_idx in range(len(df)):
        print(f"\nLigne {row_idx} :")
        row = df.iloc[row_idx]
        for col_idx, value in enumerate(row):
            content = str(value)[:80] if pd.notna(value) else "NaN"
            print(f"   Col {col_idx}: {content}")
    
    print("\n" + "-"*60)

print("\n" + "="*60)
print("💡 Identifiez lequel contient les ministères et les données")
print("="*60)

📄 ANALYSE DES 2 TABLEAUX

📊 TABLEAU 1 - 4 lignes x 11 colonnes

Contenu complet :

Ligne 0 :
T32Col 0: Première partieEVOLUTION DE L’ACTIVITE ECONOMIQUE ET FINANCIERE, 2004-2005
   Col 1: NaN
   Col 2: NaN
   Col 3: NaN
   Col 4: NaN
   Col 5: NaN
   Col 6: NaN
   Col 7: NaN
   Col 8: NaN
   Col 9: NaN
   Col 10: NaN

Ligne 1 :
   Col 0: NaN
   Col 1: 1996
   Col 2: 1997
   Col 3: 1998
   Col 4: 1999
   Col 5: 2000
   Col 6: 2001
   Col 7: 2002
   Col 8: 2003
   Col 9: 2004
   Col 10: 2005

Ligne 2 :
Compagnons de la Révolutiue.itiques
3 440,57: 18 800,7
123,7,199 15 656,9
3 4002 3: 21 457
3 253134: 28 913
64 88435: 775 128
36 242060 2 675 842
19 256631 8 695 683
24 047295326 454 601
21 988 67216 741 139
59 642989: 22 053 066

Ligne 3 :
   Col 0: TOTAL
   Col 1: 90 958,7
   Col 2: 699 314,8
   Col 3: 844 686
   Col 4: 2 154 035
   Col 5: 17 934 832
   Col 6: 52 057 902
   Col 7: 142 442 456
   Col 8: 249 344 721
   Col 9: 344 146 811
   Col 10: 472 995 414

----------------------------

In [6]:
# ============================================================
# EXTRACTION COMPLÈTE - CODE COLAB ADAPTÉ POUR VS CODE
# ============================================================

import tabula
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

# -------------------------
# PARAMÈTRES
# -------------------------
PDF_2005 = Path("../data/raw/rapport_annuel_2005.pdf")
PDF_2014 = Path("../data/raw/rapport_annuel_2014.pdf")
PDF_2023 = Path("../data/raw/rapport_annuel_2023.pdf")

PAGE_2005 = 152
PAGE_2014 = 117
PAGE_2023_153 = 153
PAGE_2023_154 = 154

AREA_2005 = [50, 10, 820, 600]
AREA_153 = [260, 20, 800, 580]
AREA_154 = [80, 20, 800, 580]

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(exist_ok=True)

OUT_INTER_2005 = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2005ok.xlsx"
OUT_INTER_2014 = OUTPUT_DIR / "Budget_Depenses_Courantes_2005_2014ok.xlsx"
OUT_INTER_2023 = OUTPUT_DIR / "Budget_Depenses_Courantes_2014_2023ok.xlsx"
OUT_WIDE = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2023_WIDEok.xlsx"
OUT_LONG = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2023_LONGok.xlsx"

# -------------------------
# FONCTIONS UTILITAIRES
# -------------------------
def to_number(val):
    if val is None:
        return np.nan
    v = str(val).strip().replace("'", "")
    if v in ["", "-", "–", "—", "nan", "NaN", "None"]:
        return np.nan
    v = v.replace("\u00A0", "").replace(" ", "").replace(",", ".")
    if not re.match(r"^-?\d+(\.\d+)?$", v):
        return np.nan
    try:
        return float(v)
    except:
        return np.nan

def clean_cells(df):
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all").reset_index(drop=True)
    df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
    return df

def drop_only_total_and_empty(df, col="Ministere"):
    keep = []
    for i, row in df.iterrows():
        name = str(row[col]).strip()
        if name == "" or name.lower() in ["nan", "none", "total"] or name.lower().startswith("source"):
            continue
        keep.append(i)
    return df.loc[keep].reset_index(drop=True)

def has_real_year_columns(df, years):
    ok_cols = sum(1 for y in years if y in df.columns and df[y].replace("", np.nan).notna().sum() >= 3)
    return ok_cols >= 5

def normalize_basic(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().replace("\n", " ").replace("\t", " ")
    s = re.sub(r"\s+", " ", s)
    s_nfkd = unicodedata.normalize("NFKD", s)
    s = "".join([c for c in s_nfkd if not unicodedata.combining(c)])
    s = s.replace("'", "'").replace("–", "-").replace("—", "-")
    s = re.sub(r"\.+$", "", s).strip()
    return s.upper()

ALIAS_TO_CANON_SHORT = [
    (r"\bSANTE\b.*\bPUBLIQUE\b", "SANTE PUBLIQUE"),
    (r"\bMINISTERE\b.*\bSANTE\b", "SANTE PUBLIQUE"),
    (r"\bSANTE\b", "SANTE PUBLIQUE"),
    (r"\bEDUCATION\b.*\bNATIONALE\b", "EDUCATION NATIONALE"),
    (r"\bENSEIGNEMENT\b.*\bPRIMAIRE\b|\bEPST\b", "EDUCATION (EPST)"),
    (r"\bENSEIGNEMENT\b.*\bSUPERIEUR\b|\bESU\b", "ENSEIGNEMENT SUPERIEUR"),
    (r"\bFINANCES\b", "FINANCES"),
    (r"\bBUDGET\b", "BUDGET"),
    (r"\bTRESOR\b", "TRESOR"),
    (r"\bDEFENSE\b", "DEFENSE"),
    (r"\bINTERIEUR\b", "INTERIEUR"),
    (r"\bSECURITE\b", "SECURITE"),
    (r"\bPOLICE\b", "POLICE"),
    (r"\bJUSTICE\b", "JUSTICE"),
    (r"\bAFFAIRES\s+ETRANGER", "AFFAIRES ETRANGERES"),
    (r"\bAGRICULT", "AGRICULTURE"),
    (r"\bTRANSPORT", "TRANSPORT"),
    (r"\bINFRASTRUCT", "INFRASTRUCTURES"),
    (r"\bTRAVAUX\s+PUBLICS\b|\bTP\b", "TRAVAUX PUBLICS"),
    (r"\bENERGIE\b", "ENERGIE"),
    (r"\bELECTRIC", "ELECTRICITE"),
    (r"\bAFFAIRES\s+SOCIALES\b", "AFFAIRES SOCIALES"),
]

def canonicalize_ministere_short(raw_name: str) -> str:
    n = normalize_basic(raw_name)
    if n in {"", "NAN", "NONE"}:
        return ""
    for pattern, canon in ALIAS_TO_CANON_SHORT:
        if re.search(pattern, n, flags=re.IGNORECASE):
            return canon
    return n

FUNCTION_RULES = [
    (r"\bSANTE\b|HOPITAL|MEDEC", "SANTE"),
    (r"\bEDUCATION\b|ENSEIGNEMENT|UNIVERSIT|\bESU\b|\bEPST\b", "EDUCATION"),
    (r"\bFINANCES\b|\bBUDGET\b|\bTRESOR\b|IMPOT|DGI|DGRAD|DGDA", "FINANCES_PUBLIQUES"),
    (r"\bDEFENSE\b|ARME(E|ES)|MILITA", "DEFENSE"),
    (r"\bINTERIEUR\b|\bSECURITE\b|\bPOLICE\b", "SECURITE_INTERIEURE"),
    (r"\bJUSTICE\b|TRIBUNAL", "JUSTICE"),
    (r"\bAFFAIRES\s+ETRANGER", "DIPLOMATIE"),
    (r"\bAGRICULT", "AGRICULTURE"),
    (r"\bTRANSPORT\b|ROUTE|VOIRIE|\bINFRASTRUCT", "TRANSPORT_INFRA"),
    (r"\bENERGIE\b|ELECTRIC", "ENERGIE"),
]

def assign_function(ministere_canonique: str) -> str:
    n = normalize_basic(ministere_canonique)
    if n == "":
        return "AUTRE"
    for pattern, func in FUNCTION_RULES:
        if re.search(pattern, n, flags=re.IGNORECASE):
            return func
    return "AUTRE"

def enrich_ministere_columns(df, col="Ministere"):
    df = df.copy()
    df["Ministere_raw"] = df[col].astype(str)
    df["Ministere_canonique"] = df["Ministere_raw"].apply(canonicalize_ministere_short)
    df["Fonction"] = df["Ministere_canonique"].apply(assign_function)
    df["Ministere_key"] = df["Ministere_canonique"].apply(normalize_basic)
    df[col] = df["Ministere_canonique"]
    return df

def drop_non_ministere_rows_2023(df, col="Ministere"):
    rubric_letter = re.compile(r"^\s*[A-Z]\.\s+")
    fiscal_keywords = [r"\bimpots?\b", r"\btaxe(s)?\b", r"\bdroits?\b", r"\baccises?\b",
                      r"\bvehicules?\b", r"\bimmatriculation\b", r"\bdouan(e|es)\b"]
    fiscal_regex = re.compile("|".join(fiscal_keywords), flags=re.IGNORECASE)
    
    def is_bad(name: str) -> bool:
        if name is None:
            return True
        raw = str(name).strip()
        if raw == "" or raw.lower() in ["nan", "none"]:
            return True
        if rubric_letter.search(raw):
            return True
        n = normalize_basic(raw)
        if fiscal_regex.search(n):
            return True
        return False
    
    keep = [i for i, row in df.iterrows() if not is_bad(row[col])]
    return df.loc[keep].reset_index(drop=True)

# -------------------------
# EXTRACTION (Méthode Colab)
# -------------------------
def tabula_read(pdf_path, page, lattice=False, stream=False, area=None, guess=True):
    return tabula.read_pdf(
        str(pdf_path),
        pages=page,
        multiple_tables=True,
        lattice=lattice,
        stream=stream,
        area=area,
        guess=guess,
        pandas_options={"header": None}
    )

def pick_best_candidate(dfs, year_start=1996, year_end=2005):
    if not dfs:
        return None
    years = [str(y) for y in range(year_start, year_end + 1)]
    best_df, best_score = None, -1
    for d in dfs:
        df = clean_cells(d)
        first_rows = " ".join(df.head(5).astype(str).fillna("").values.flatten().tolist())
        year_hits = sum(1 for y in years if y in first_rows)
        score = df.shape[1] * 2 + year_hits * 5
        if score > best_score:
            best_score, best_df = score, df
    return best_df

def reshape_fixed_years(raw_df, year_start, year_end):
    years = [str(y) for y in range(year_start, year_end + 1)]
    df = clean_cells(raw_df)
    expected_cols = 1 + len(years)
    if df.shape[1] < expected_cols:
        raise RuntimeError("Pas assez de colonnes.")
    df = df.iloc[:, :expected_cols].copy()
    df.columns = ["Ministere"] + years
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    for y in years:
        df[y] = df[y].apply(to_number)
    return df.reset_index(drop=True)

def extract_2005(pdf_path, page):
    years = [str(y) for y in range(1996, 2006)]
    
    # Tentative 1: lattice
    dfs = tabula_read(pdf_path, page, lattice=True, stream=False, area=None, guess=True)
    cand = pick_best_candidate(dfs)
    if cand is not None:
        try:
            df = reshape_fixed_years(cand, 1996, 2005)
            if has_real_year_columns(df, years):
                return df
        except:
            pass
    
    # Tentative 2: stream
    dfs = tabula_read(pdf_path, page, lattice=False, stream=True, area=None, guess=True)
    cand = pick_best_candidate(dfs)
    if cand is not None:
        try:
            df = reshape_fixed_years(cand, 1996, 2005)
            if has_real_year_columns(df, years):
                return df
        except:
            pass
    
    # Tentative 3: stream + AREA
    dfs = tabula_read(pdf_path, page, lattice=False, stream=True, area=AREA_2005, guess=False)
    cand = pick_best_candidate(dfs)
    if cand is not None:
        df = reshape_fixed_years(cand, 1996, 2005)
        if has_real_year_columns(df, years):
            return df
    
    raise RuntimeError("❌ Extraction 2005 échouée")

def extract_2014(pdf_path, page):
    dfs = tabula_read(pdf_path, page, lattice=True, stream=False, area=None, guess=True)
    if not dfs:
        raise RuntimeError("❌ Extraction 2014 échouée")
    raw = max(dfs, key=lambda d: d.shape[1])
    df = reshape_fixed_years(raw, 2005, 2014)
    return df

def extract_2023(pdf_path):
    dfs_153 = tabula.read_pdf(str(pdf_path), pages=PAGE_2023_153, area=AREA_153,
                             lattice=True, multiple_tables=False, pandas_options={"header": None})
    dfs_154 = tabula.read_pdf(str(pdf_path), pages=PAGE_2023_154, area=AREA_154,
                             lattice=True, multiple_tables=False, pandas_options={"header": None})
    
    if not dfs_153 or not dfs_154:
        raise RuntimeError("❌ Extraction 2023 échouée")
    
    r153, r154 = dfs_153[0], dfs_154[0]
    d153, d154 = clean_cells(r153), clean_cells(r154)
    df = pd.concat([d153, d154], axis=0, ignore_index=True)
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all").reset_index(drop=True)
    df = df.iloc[:, :11].copy()
    df.columns = ["Ministere"] + [str(y) for y in range(2014, 2024)]
    df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = drop_only_total_and_empty(df, "Ministere")
    df = drop_non_ministere_rows_2023(df, "Ministere")
    for y in range(2014, 2024):
        df[str(y)] = df[str(y)].apply(to_number)
    return df.reset_index(drop=True)

def merge_two(left, right, prefer_right_years=None):
    years_left = sorted([c for c in left.columns if re.match(r"^(19|20)\d{2}$", str(c))], key=lambda x: int(x))
    years_right = sorted([c for c in right.columns if re.match(r"^(19|20)\d{2}$", str(c))], key=lambda x: int(x))
    
    base_left = ["Ministere", "Ministere_key"]
    base_right = ["Ministere", "Ministere_key"]
    if "Fonction" in left.columns: base_left.append("Fonction")
    if "Fonction" in right.columns: base_right.append("Fonction")
    
    m = pd.merge(left[base_left + years_left], right[base_right + years_right],
                on="Ministere_key", how="outer", suffixes=("_L", "_R"))
    
    m["Ministere"] = m.get("Ministere_R").where(
        m.get("Ministere_R").notna() & (m.get("Ministere_R").astype(str).str.strip() != ""),
        m.get("Ministere_L"))
    m.drop(columns=["Ministere_L", "Ministere_R"], inplace=True, errors="ignore")
    
    if "Fonction_L" in m.columns or "Fonction_R" in m.columns:
        m["Fonction"] = m.get("Fonction_R").where(
            m.get("Fonction_R").notna() & (m.get("Fonction_R").astype(str).str.strip() != ""),
            m.get("Fonction_L"))
        m.drop(columns=["Fonction_L", "Fonction_R"], inplace=True, errors="ignore")
    
    if prefer_right_years:
        for y in prefer_right_years:
            yl, yr = f"{y}_L", f"{y}_R"
            if yl in m.columns and yr in m.columns:
                m[y] = m[yr].combine_first(m[yl])
                m.drop(columns=[yl, yr], inplace=True)
    
    for y in years_left:
        col = f"{y}_L"
        if col in m.columns and y not in m.columns:
            m.rename(columns={col: y}, inplace=True)
    for y in years_right:
        col = f"{y}_R"
        if col in m.columns and y not in m.columns:
            m.rename(columns={col: y}, inplace=True)
    
    return m

# -------------------------
# EXÉCUTION
# -------------------------
print("🔎 EXTRACTION DES 3 RAPPORTS BCC")
print("="*60)

print("\n📄 1/3 Rapport 2005...")
df2005 = extract_2005(PDF_2005, PAGE_2005)
df2005 = enrich_ministere_columns(df2005, col="Ministere")
print(f"✅ 2005: {df2005.shape}")

print("\n📄 2/3 Rapport 2014...")
df2014 = extract_2014(PDF_2014, PAGE_2014)
df2014 = enrich_ministere_columns(df2014, col="Ministere")
print(f"✅ 2014: {df2014.shape}")

print("\n📄 3/3 Rapport 2023...")
df2023 = extract_2023(PDF_2023)
df2023 = enrich_ministere_columns(df2023, col="Ministere")
print(f"✅ 2023: {df2023.shape}")

# Sauvegardes intermédiaires
print("\n📊 Sauvegarde des fichiers intermédiaires...")
cols_2005 = ["Ministere", "Fonction"] + [str(y) for y in range(1996, 2006)]
cols_2014 = ["Ministere", "Fonction"] + [str(y) for y in range(2005, 2015)]
cols_2023 = ["Ministere", "Fonction"] + [str(y) for y in range(2014, 2024)]

df2005[cols_2005].to_excel(OUT_INTER_2005, index=False, engine='openpyxl')
df2014[cols_2014].to_excel(OUT_INTER_2014, index=False, engine='openpyxl')
df2023[cols_2023].to_excel(OUT_INTER_2023, index=False, engine='openpyxl')

print(f"   ✅ {OUT_INTER_2005.name}")
print(f"   ✅ {OUT_INTER_2014.name}")
print(f"   ✅ {OUT_INTER_2023.name}")

# Consolidation
print("\n🔗 Consolidation...")
m1 = merge_two(df2005, df2014, prefer_right_years=["2005"])
m2 = merge_two(m1, df2023, prefer_right_years=["2014"])

all_years = sorted([c for c in m2.columns if re.match(r"^(19|20)\d{2}$", str(c))], key=lambda x: int(x))
wide_cols = ["Ministere", "Fonction"] + all_years
wide = m2[wide_cols].copy()
wide = wide[wide["Ministere"].notna() & (wide["Ministere"].astype(str).str.strip() != "")].reset_index(drop=True)
wide.to_excel(OUT_WIDE, index=False, engine='openpyxl')
print(f"   ✅ WIDE: {OUT_WIDE.name}")

long_df = wide.melt(id_vars=["Ministere", "Fonction"], var_name="Annee", value_name="Budget_Depense_Courante")
long_df["Annee"] = long_df["Annee"].astype(int)
long_df.to_excel(OUT_LONG, index=False, engine='openpyxl')
print(f"   ✅ LONG: {OUT_LONG.name}")

print("\n" + "="*60)
print("📈 RÉSUMÉ")
print("="*60)
print(f"Période           : {long_df['Annee'].min()} → {long_df['Annee'].max()}")
print(f"Ministères        : {long_df['Ministere'].nunique()}")
print(f"Fonctions         : {long_df['Fonction'].nunique()}")
print(f"Observations      : {len(long_df):,}")
print(f"Valeurs manquantes: {long_df['Budget_Depense_Courante'].isna().mean()*100:.1f}%")
print("="*60)

long_df.head(10)

🔎 EXTRACTION DES 3 RAPPORTS BCC

📄 1/3 Rapport 2005...


C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\21138

✅ 2005: (64, 15)

📄 2/3 Rapport 2014...


Got stderr: févr. 10, 2026 3:59:42 PM org.apache.pdfbox.pdmodel.font.FileSystemFontProvider loadDiskCache
AVERTISSEMENT: New fonts found, font cache will be re-built
févr. 10, 2026 3:59:42 PM org.apache.pdfbox.pdmodel.font.FileSystemFontProvider <init>
AVERTISSEMENT: Building on-disk font cache, this may take a while
févr. 10, 2026 3:59:44 PM org.apache.pdfbox.pdmodel.font.FileSystemFontProvider <init>
AVERTISSEMENT: Finished building on-disk font cache, found 361 fonts
févr. 10, 2026 3:59:44 PM org.apache.pdfbox.pdmodel.font.PDTrueTypeFont <init>
AVERTISSEMENT: Using fallback font 'ArialMT' for 'AAMGYD+Helvetica'

C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())


✅ 2014: (2, 15)

📄 3/3 Rapport 2023...


C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())


✅ 2023: (58, 15)

📊 Sauvegarde des fichiers intermédiaires...
   ✅ Budget_Depenses_Courantes_1996_2005ok.xlsx
   ✅ Budget_Depenses_Courantes_2005_2014ok.xlsx
   ✅ Budget_Depenses_Courantes_2014_2023ok.xlsx

🔗 Consolidation...
   ✅ WIDE: Budget_Depenses_Courantes_1996_2023_WIDEok.xlsx
   ✅ LONG: Budget_Depenses_Courantes_1996_2023_LONGok.xlsx

📈 RÉSUMÉ
Période           : 1996 → 2023
Ministères        : 72
Fonctions         : 10
Observations      : 2,044
Valeurs manquantes: 48.4%


,Ministere,Fonction,Annee,Budget_Depense_Courante
0,ADMINISTRATION DU TERRITOIRE,AUTRE,1996,147.5
1,AFFAIRES ETRANGERES,DIPLOMATIE,1996,184.4
2,AFFAIRES FONCIERES,AUTRE,1996,143.6
3,AFFAIRES SOCIALES,AUTRE,1996,59.2
4,AGRICULTURE,AGRICULTURE,1996,69.4
5,ANCIENS COMBATTANTS,AUTRE,1996,36.9
6,ASSEMBLEE NATIONALE & H.C.R,AUTRE,1996,13301.9
7,ASSEMBLEE NATIONALE & SENAT,AUTRE,1996,NaN
8,AUTRES SERVICES,AUTRE,1996,NaN
9,AUTRES SERVICES (PPTE),AUTRE,1996,NaN


In [11]:
# ============================================================
# EXTRACTION 2014 - AVEC CAMELOT
# ============================================================

print("🔧 EXTRACTION RAPPORT 2014 - AVEC CAMELOT")
print("="*60)

try:
    import camelot
    
    # Extraction avec camelot (comme sur Colab)
    tables = camelot.read_pdf(
        str(PDF_2014),
        pages=str(PAGE_2014),
        flavor="stream",
        strip_text="\n",
        split_text=True
    )
    
    print(f"✅ {tables.n} tableau(x) détecté(s)\n")
    
    if tables.n > 0:
        # Prendre le tableau avec le plus de colonnes
        best_table = max(tables, key=lambda t: t.df.shape[1])
        df_raw = best_table.df
        
        print(f"📊 Meilleur tableau : {df_raw.shape}")
        print(f"\n   Aperçu brut (5 premières lignes) :")
        print(df_raw.head(5).to_string())
        print()
        
        # Nettoyer et structurer
        df = df_raw.copy()
        df = df.dropna(axis=0, how='all').reset_index(drop=True)
        
        # Renommer colonnes
        years = [str(y) for y in range(2005, 2015)]
        
        if df.shape[1] >= 11:
            df = df.iloc[:, :11].copy()
            df.columns = ["Ministere"] + years
            
            print(f"   ✅ Colonnes renommées : {list(df.columns)}\n")
            
            # Nettoyer Ministere
            df["Ministere"] = df["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
            df = df[df["Ministere"].notna()].copy()
            df = df[~df["Ministere"].str.lower().isin(['nan', 'none', '', 'total'])].copy()
            df = df[~df["Ministere"].str.lower().str.startswith('source')].copy()
            df = df.reset_index(drop=True)
            
            # Conversion numérique
            for year in years:
                df[year] = df[year].apply(to_number)
            
            # Enrichir
            df = enrich_ministere_columns(df, col="Ministere")
            
            print(f"   ✅ Données traitées : {df.shape}\n")
            
            # Statistiques
            print(f"   📊 Valeurs manquantes par année :")
            for year in years:
                missing = df[year].isna().sum()
                total = len(df)
                pct = (missing / total) * 100
                status = "✅" if pct < 50 else "⚠️"
                print(f"      {status} {year} : {missing}/{total} ({pct:.1f}%)")
            
            # Aperçu
            print(f"\n   📋 Aperçu (10 premières lignes) :")
            cols_check = ['Ministere', 'Fonction', '2005', '2007', '2008', '2012', '2014']
            print(df[cols_check].head(10).to_string())
            
            # Sauvegarder
            cols_2014 = ["Ministere", "Fonction"] + years
            try:
                df[cols_2014].to_excel(OUT_INTER_2014, index=False, engine='openpyxl')
                print(f"\n   ✅ Fichier sauvegardé : {OUT_INTER_2014.name}")
                df2014 = df
                print(f"   ✅ Variable df2014 mise à jour")
            except PermissionError:
                alt_file = OUTPUT_DIR / "Budget_Depenses_Courantes_2005_2014_CAMELOT.xlsx"
                df[cols_2014].to_excel(alt_file, index=False, engine='openpyxl')
                print(f"\n   ✅ Sauvegardé : {alt_file.name}")
                df2014 = df
        else:
            print(f"   ❌ Pas assez de colonnes : {df.shape[1]}")
    else:
        print("   ❌ Aucun tableau détecté")
        
except ImportError:
    print("❌ Camelot non installé")
    print("   Installez avec : pip install camelot-py[cv]")
    
except Exception as e:
    print(f"❌ ERREUR : {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

🔧 EXTRACTION RAPPORT 2014 - AVEC CAMELOT


c:\Users\LEGION\Music\Memoire Master\venv\lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


✅ 1 tableau(x) détecté(s)

📊 Meilleur tableau : (64, 11)

   Aperçu brut (5 premières lignes) :
                             0         1         2         3         4         5          6          7          8          9          10
0                                                     (en mil   lions d    e CDF)                                                       
1                                    2005      2006      2007      2008      2009       2010       2011       2012       2013       2014
2     I.Institutions politiques  22 053,1  32 906,0  42 789,8  48 989,9  69 158,7  169 907,7  192 297,5  318 510,0  308 747,9  345 889,1
3  Présidence de la République.   9 762,4  14 751,2  11 185,9  13 320,0  16 542,7   29 539,0   31 370,7   50 000,0   53 000,0   63 191,5
4   Assemblée Nationale & Sénat   5 003,1   4 829,0  22 480,8  24 349,6  32 133,3  104 810,3   99 654,7  122 850,0  132 098,9  142 809,9

   ✅ Colonnes renommées : ['Ministere', '2005', '2006', '2007', '2008', '2009', '

In [ ]:
# ============================================================
# NOUVEAU REGROUPEMENT EN FONCTIONS (COFOG-inspired)
# ============================================================

print("🔧 PROPOSITION DE NOUVEAU REGROUPEMENT")
print("="*60)

# Nouveau mapping COHÉRENT
NOUVEAU_MAPPING_FONCTIONS = {
    # 1. SERVICES GÉNÉRAUX
    "SERVICES_GENERAUX": [
        r"\bPRESIDENCE\b",
        r"\bPRIMATURE\b",
        r"\bASSEMBLEE\b",
        r"\bSENAT\b",
        r"\bPARLEMENT\b",
        r"\bINSTITUTIONS\s+POLITIQUES\b",
        r"\bCOMPAGNONS\b",
        r"\bBUREAU\b.*\bMINISTRE\b",
        r"\bREFORMES\s+INSTITUTIONNELLES\b",
        r"\bRELATIONS\b.*\bPARTIS\b",
        r"\bCOMITE\s+DIRECTEUR\b",
        r"\bDIALOGUE\b.*\bCONGOLAIS\b",
    ],
    
    # 2. DÉFENSE & SÉCURITÉ
    "DEFENSE_SECURITE": [
        r"\bDEFENSE\b",
        r"\bARME(E|ES)\b",
        r"\bMILITAIRE\b",
        r"\bCOMBATTANT\b",
        r"\bINTERIEUR\b",
        r"\bSECURITE\b",
        r"\bPOLICE\b",
        r"\bSERVICES\s+DE\s+SECURITE\b",
        r"\bORGANISMES\s+AUXILIAIRES\b",
    ],
    
    # 3. ORDRE PUBLIC & JUSTICE
    "JUSTICE": [
        r"\bJUSTICE\b",
        r"\bMAGISTRATURE\b",
        r"\bTRIBUNAUX\b",
        r"\bCOURS\b",
        r"\bDROITS\s+HUMAINS\b",
        r"\bDROITS\b.*\bLIBERTES\b.*\bCITOYEN\b",
    ],
    
    # 4. AFFAIRES ÉCONOMIQUES
    "AFFAIRES_ECONOMIQUES": [
        r"\bECONOMIE\b",
        r"\bPLAN\b",
        r"\bCOMMERCE\b",
        r"\bINDUSTRIE\b",
        r"\bARTISANAT\b",
        r"\bPME\b",
        r"\bPETITES\s+ET\s+MOYENNES\s+ENTREPRISES\b",
        r"\bPORTEFEUILLE\b",
        r"\bPARTICIPATIONS\b.*\bPRIVATISATIONS\b",
    ],
    
    # 5. ENVIRONNEMENT
    "ENVIRONNEMENT": [
        r"\bENVIRONNEMENT\b",
        r"\bCONSERVATION\b.*\bNATURE\b",
        r"\bFORETS\b",
    ],
    
    # 6. LOGEMENT & ÉQUIPEMENTS COLLECTIFS
    "LOGEMENT_AMENAGEMENT": [
        r"\bURBANISME\b",
        r"\bHABITAT\b",
        r"\bAMENAGEMENT\b.*\bTERRITOIRE\b",
        r"\bTRAVAUX\s+PUBLICS\b",
        r"\bINFRASTRUCTURE\b",
        r"\bRECONSTRUCTION\b",
    ],
    
    # 7. SANTÉ
    "SANTE": [
        r"\bSANTE\b",
        r"\bMEDEC\b",
        r"\bHOPITAL\b",
        r"\bHYGIENE\b",
        r"\bSIDA\b",
    ],
    
    # 8. LOISIRS, CULTURE & RELIGION
    "CULTURE_LOISIRS": [
        r"\bCULTURE\b",
        r"\bARTS\b",
        r"\bSPORTS\b",
        r"\bJEUNESSE\b",
        r"\bLOISIRS\b",
        r"\bTOURISME\b",
        r"\bHOTELLERIE\b",
    ],
    
    # 9. ÉDUCATION
    "EDUCATION": [
        r"\bEDUCATION\b",
        r"\bENSEIGNEMENT\b",
        r"\bUNIVERSIT\b",
        r"\bEPST\b",
        r"\bESU\b",
        r"\bRECHERCHE\s+SCIENTIFIQUE\b",
        r"\bALPHABETISATION\b",
    ],
    
    # 10. PROTECTION SOCIALE
    "PROTECTION_SOCIALE": [
        r"\bAFFAIRES\s+SOCIALES\b",
        r"\bGENRE\b",
        r"\bFAMILLE\b",
        r"\bFEMININE\b",
        r"\bSOLIDARITE\b",
        r"\bACTION\s+HUMANITAIRE\b",
        r"\bTRAVAIL\b",
        r"\bPREVOYANCE\s+SOCIALES\b",
        r"\bFONCTION\s+PUBLIQUE\b",
        r"\bEMPLOI\b",
    ],
    
    # 11. AGRICULTURE & RESSOURCES
    "AGRICULTURE_PECHE": [
        r"\bAGRICULTURE\b",
        r"\bPECHE\b",
        r"\bELEVAGE\b",
        r"\bDEVELOPPEMENT\s+RURAL\b",
    ],
    
    # 12. MINES & ÉNERGIE
    "MINES_ENERGIE": [
        r"\bMINES\b",
        r"\bHYDROCARBURE\b",
        r"\bPETROLE\b",
        r"\bENERGIE\b",
        r"\bELECTRIC\b",
        r"\bRESSOURCES\s+HYDRAULIQUES\b",
    ],
    
    # 13. TRANSPORTS
    "TRANSPORTS": [
        r"\bTRANSPORT\b",
        r"\bCOMMUNICATIONS\b",
        r"\bVOIES\s+DE\s+COMMUNICATION\b",
        r"\bROUTE\b",
        r"\bVOIRIE\b",
    ],
    
    # 14. COMMUNICATION & MÉDIAS
    "COMMUNICATION_MEDIAS": [
        r"\bINFORMATION\b",
        r"\bPRESSE\b",
        r"\bMEDIAS\b",
        r"\bPOSTES\b",
        r"\bTELEGRAPHES\b",
        r"\bTELECOMMUNICATIONS\b",
    ],
    
    # 15. AFFAIRES ÉTRANGÈRES
    "AFFAIRES_ETRANGERES": [
        r"\bAFFAIRES\s+ETRANGER\b",
        r"\bRELATIONS\s+EXTERIEURES\b",
        r"\bCOOPERATION\s+INTERNATIONALE\b",
        r"\bDIPLOMATIE\b",
    ],
    
    # 16. FINANCES PUBLIQUES
    "FINANCES_PUBLIQUES": [
        r"\bFINANCES\b",
        r"\bBUDGET\b",
        r"\bTRESOR\b",
        r"\bIMPOT\b",
        r"\bDGI\b",
        r"\bDGRAD\b",
        r"\bDGDA\b",
        r"\bDOUANES\b",
        r"\bREGIE\s+FINANCIERE\b",
    ],
    
    # 17. AFFAIRES FONCIÈRES
    "AFFAIRES_FONCIERES": [
        r"\bAFFAIRES\s+FONCIERES\b",
        r"\bCADASTRE\b",
        r"\bTITRES\s+FONCIERS\b",
    ],
    
    # 18. ADMINISTRATION TERRITORIALE
    "ADMINISTRATION_TERRITOIRE": [
        r"\bADMINISTRATION\s+DU\s+TERRITOIRE\b",
        r"\bDECENTRALISATION\b",
        r"\bAFFAIRES\s+INTERIEURES\b",
    ],
}

def nouvelle_attribution_fonction(ministere: str) -> str:
    """Attribution de fonction selon le nouveau mapping"""
    if not ministere or pd.isna(ministere):
        return "NON_CLASSIFIE"
    
    n = normalize_basic(ministere)
    
    if n == "":
        return "NON_CLASSIFIE"
    
    # Tester chaque fonction
    for fonction, patterns in NOUVEAU_MAPPING_FONCTIONS.items():
        for pattern in patterns:
            if re.search(pattern, n, flags=re.IGNORECASE):
                return fonction
    
    return "NON_CLASSIFIE"

# Tester le nouveau regroupement
print("\n🧪 TEST DU NOUVEAU REGROUPEMENT")
print("="*60)

# Appliquer sur tous les ministères
all_ministeres['Nouvelle_Fonction'] = all_ministeres['Ministere'].apply(nouvelle_attribution_fonction)

# Statistiques
print("\n📊 STATISTIQUES DU NOUVEAU REGROUPEMENT")
print("="*60)

nouveau_stats = all_ministeres.groupby('Nouvelle_Fonction').size().sort_values(ascending=False)

total = len(all_ministeres)
for fonction, count in nouveau_stats.items():
    pct = (count / total) * 100
    status = "✅" if pct < 30 else "⚠️" if pct < 50 else "❌"
    print(f"{status} {fonction:30s} : {count:3d} ministères ({pct:5.1f}%)")

# Comparer ancien vs nouveau
print("\n" + "="*60)
print("📊 COMPARAISON ANCIEN vs NOUVEAU")
print("="*60)

ancien_autre = (all_ministeres['Fonction'] == 'AUTRE').sum()
nouveau_nc = (all_ministeres['Nouvelle_Fonction'] == 'NON_CLASSIFIE').sum()

print(f"Ancien système - AUTRE       : {ancien_autre}/{total} ({ancien_autre/total*100:.1f}%)")
print(f"Nouveau système - NON_CLASSIFIE : {nouveau_nc}/{total} ({nouveau_nc/total*100:.1f}%)")
print(f"\n✅ Amélioration : {ancien_autre - nouveau_nc} ministères mieux classifiés")

# Afficher détails par fonction
print("\n" + "="*60)
print("📋 DÉTAIL PAR FONCTION (NOUVEAU SYSTÈME)")
print("="*60)

for fonction in sorted(all_ministeres['Nouvelle_Fonction'].unique()):
    if fonction == "NON_CLASSIFIE":
        continue
    
    ministeres = all_ministeres[all_ministeres['Nouvelle_Fonction'] == fonction]['Ministere'].tolist()
    print(f"\n🏛️ {fonction} ({len(ministeres)} ministères)")
    print("-" * 60)
    for i, min_name in enumerate(ministeres, 1):
        print(f"   {i}. {min_name}")

# Afficher les NON_CLASSIFIES pour analyse
if nouveau_nc > 0:
    print(f"\n⚠️ NON_CLASSIFIÉS ({nouveau_nc} ministères)")
    print("-" * 60)
    non_class = all_ministeres[all_ministeres['Nouvelle_Fonction'] == 'NON_CLASSIFIE']['Ministere'].tolist()
    for i, min_name in enumerate(non_class, 1):
        print(f"   {i}. {min_name}")

print("\n" + "="*60)

🔧 PROPOSITION DE NOUVEAU REGROUPEMENT

🧪 TEST DU NOUVEAU REGROUPEMENT

📊 STATISTIQUES DU NOUVEAU REGROUPEMENT
✅ NON_CLASSIFIE                  :  18 ministères ( 25.0%)
✅ SERVICES_GENERAUX              :  12 ministères ( 16.7%)
✅ AFFAIRES_ECONOMIQUES           :   8 ministères ( 11.1%)
✅ JUSTICE                        :   4 ministères (  5.6%)
✅ PROTECTION_SOCIALE             :   4 ministères (  5.6%)
✅ CULTURE_LOISIRS                :   3 ministères (  4.2%)
✅ DEFENSE_SECURITE               :   3 ministères (  4.2%)
✅ LOGEMENT_AMENAGEMENT           :   3 ministères (  4.2%)
✅ MINES_ENERGIE                  :   3 ministères (  4.2%)
✅ AGRICULTURE_PECHE              :   2 ministères (  2.8%)
✅ COMMUNICATION_MEDIAS           :   2 ministères (  2.8%)
✅ EDUCATION                      :   2 ministères (  2.8%)
✅ FINANCES_PUBLIQUES             :   2 ministères (  2.8%)
✅ AFFAIRES_ETRANGERES            :   1 ministères (  1.4%)
✅ ADMINISTRATION_TERRITOIRE      :   1 ministères (  1.4%)
✅ ENV

In [14]:
# ============================================================
# ANALYSE DES NON_CLASSIFIÉS + PROPOSITIONS
# ============================================================

print("🔍 ANALYSE DES 18 MINISTÈRES NON_CLASSIFIÉS")
print("="*60)

non_class = all_ministeres[all_ministeres['Nouvelle_Fonction'] == 'NON_CLASSIFIE'].copy()

print(f"\nListe des {len(non_class)} ministères à classifier :\n")

for i, row in non_class.iterrows():
    print(f"{i+1:2d}. {row['Ministere']}")

print("\n" + "="*60)
print("💡 PROPOSITIONS DE CLASSIFICATION")
print("="*60)

# Propositions manuelles pour les cas ambigus
propositions = {
    "III.AUTRES SERVICES": "SERVICES_GENERAUX",
    "DETTE PUBLIQUE": "FINANCES_PUBLIQUES",
    "VILLES ET PROVINCES": "ADMINISTRATION_TERRITOIRE",
    "DEPENSES CENTRALISEES": "FINANCES_PUBLIQUES",
    "BUDGETS ANNEXES": "FINANCES_PUBLIQUES",
    "DEPENSES POUR ORDRE": "FINANCES_PUBLIQUES",
    "AUTRES SERVICES (PPTE)": "FINANCES_PUBLIQUES",
    "II.MINISTERES": "SERVICES_GENERAUX",
    "AUTRES": "SERVICES_GENERAUX",
    "DEPENSES EXCEPTIONNELLES": "FINANCES_PUBLIQUES",
}

print("\nPropositions de reclassification :")
for ministere, fonction_proposee in propositions.items():
    print(f"   • {ministere:40s} → {fonction_proposee}")

print("\n💬 Validez-vous ces propositions ?")
print("   Si oui, nous mettrons à jour le mapping définitif.")

🔍 ANALYSE DES 18 MINISTÈRES NON_CLASSIFIÉS

Liste des 18 ministères à classifier :

56. DEPENSES COMMUNES (5)
 6. AUTRES(3)
64. AUTRES SERVICES (PPTE)
62. BUDGETS ANNEXES
61. DEPENSES CENTRALISEES (2)
60. VILLES ET PROVINCES
59. DETTE PUBLIQUE
58. III.AUTRES SERVICES
57. MINISTERES NON IDENTIFIES
63. DEPENSES POUR ORDRE
 6. SERV. TECHN. DE LA PRES
 8. AUTRES(4)
12. II.MINISTERES
17. ANCIENS COMBATTANTS
59. DEPENSES EXCEPTIONNELLES
60. AUTRES SERVICES
25. TERRITOIRE
14. AFFAIRES ETRANGERES

💡 PROPOSITIONS DE CLASSIFICATION

Propositions de reclassification :
   • III.AUTRES SERVICES                      → SERVICES_GENERAUX
   • DETTE PUBLIQUE                           → FINANCES_PUBLIQUES
   • VILLES ET PROVINCES                      → ADMINISTRATION_TERRITOIRE
   • DEPENSES CENTRALISEES                    → FINANCES_PUBLIQUES
   • BUDGETS ANNEXES                          → FINANCES_PUBLIQUES
   • DEPENSES POUR ORDRE                      → FINANCES_PUBLIQUES
   • AUTRES SERVICES (PPTE) 

In [15]:
# ============================================================
# MAPPING FINAL DÉFINITIF DES FONCTIONS
# ============================================================

print("🔧 MAPPING FINAL DÉFINITIF")
print("="*60)

MAPPING_FONCTIONS_DEFINITIF = {
    # 1. PRÉSIDENCE
    "PRESIDENCE": [
        r"\bPRESIDENCE\b",
        r"\bSERV\.\s*TECHN\.\s*DE\s*LA\s*PRES\b",
        r"\bSERVICES\s+TECHNIQUES\b.*\bPRESIDENCE\b",
    ],
    
    # 2. PRIMATURE
    "PRIMATURE": [
        r"\bPRIMATURE\b",
        r"\bBUREAU\b.*\b(1ER|PREMIER)\s+MINISTRE\b",
    ],
    
    # 3. PARLEMENT
    "PARLEMENT": [
        r"\bASSEMBLEE\b",
        r"\bSENAT\b",
        r"\bPARLEMENT\b",
        r"\bRELATIONS\b.*\bPARLEMENT\b",
    ],
    
    # 4. SERVICES GÉNÉRAUX
    "SERVICES_GENERAUX": [
        r"\bINSTITUTIONS\s+POLITIQUES\b",
        r"\bCOMPAGNONS\b",
        r"\bREFORMES\s+INSTITUTIONNELLES\b",
        r"\bRELATIONS\b.*\bPARTIS\b",
        r"\bCOMITE\s+DIRECTEUR\b",
        r"\bDIALOGUE\b.*\bCONGOLAIS\b",
        r"\bAUTRES\b",
        r"\bII\.MINISTERES\b",
        r"\bIII\.AUTRES\s+SERVICES\b",
        r"\bAUTRES\s+SERVICES\b",
        r"\bMINISTERES\s+NON\s+IDENTIFIES\b",
        r"\bORGANISMES\s+AUXILIAIRES\b",
    ],
    
    # 5. DÉFENSE & SÉCURITÉ
    "DEFENSE_SECURITE": [
        r"\bDEFENSE\b",
        r"\bARME(E|ES)\b",
        r"\bMILITAIRE\b",
        r"\bANCIENS\s+COMBATTANTS\b",
        r"\bINTERIEUR\b",
        r"\bSECURITE\b",
        r"\bPOLICE\b",
        r"\bSERVICES\s+DE\s+SECURITE\b",
    ],
    
    # 6. JUSTICE
    "JUSTICE": [
        r"\bJUSTICE\b",
        r"\bMAGISTRATURE\b",
        r"\bTRIBUNAUX\b",
        r"\bCOURS\b",
        r"\bDROITS\s+HUMAINS\b",
        r"\bDROITS\b.*\bLIBERTES\b.*\bCITOYEN\b",
    ],
    
    # 7. ÉCONOMIE
    "ECONOMIE": [
        r"\bECONOMIE\b",
        r"\bPLAN\b",
        r"\bCOMMERCE\b",
        r"\bINDUSTRIE\b",
        r"\bDETTE\s+PUBLIQUE\b",
    ],
    
    # 8. PORTEFEUILLE ET PME (nouvelle fonction)
    "PORTEFEUILLE_PME": [
        r"\bARTISANAT\b",
        r"\bPME\b",
        r"\bPETITES\s+ET\s+MOYENNES\s+ENTREPRISES\b",
        r"\bPORTEFEUILLE\b",
        r"\bPARTICIPATIONS\b.*\bPRIVATISATIONS\b",
    ],
    
    # 9. FINANCES PUBLIQUES
    "FINANCES_PUBLIQUES": [
        r"\bFINANCES\b",
        r"\bBUDGET\b",
        r"\bTRESOR\b",
        r"\bIMPOT\b",
        r"\bDGI\b",
        r"\bDGRAD\b",
        r"\bDGDA\b",
        r"\bDOUANES\b",
        r"\bREGIE\s+FINANCIERE\b",
        r"\bAUTRES\s+SERVICES\s*\(PPTE\)\b",
    ],
    
    # 10. DÉPENSES EXCEPTIONNELLES (nouvelle fonction)
    "DEPENSES_EXCEPTIONNELLES": [
        r"\bDEPENSES\s+CENTRALISEES\b",
        r"\bBUDGETS\s+ANNEXES\b",
        r"\bDEPENSES\s+POUR\s+ORDRE\b",
        r"\bDEPENSES\s+EXCEPTIONNELLES\b",
        r"\bDEPENSES\s+COMMUNES\b",
    ],
    
    # 11. ENVIRONNEMENT
    "ENVIRONNEMENT": [
        r"\bENVIRONNEMENT\b",
        r"\bCONSERVATION\b.*\bNATURE\b",
        r"\bFORETS\b",
    ],
    
    # 12. URBANISME ET HABITAT
    "URBANISME_HABITAT": [
        r"\bURBANISME\b",
        r"\bHABITAT\b",
        r"\bAMENAGEMENT\b.*\bTERRITOIRE\b",
    ],
    
    # 13. TRAVAUX PUBLICS
    "TRAVAUX_PUBLICS": [
        r"\bTRAVAUX\s+PUBLICS\b",
        r"\bINFRASTRUCTURE\b",
        r"\bRECONSTRUCTION\b",
    ],
    
    # 14. SANTÉ
    "SANTE": [
        r"\bSANTE\b",
        r"\bMEDEC\b",
        r"\bHOPITAL\b",
        r"\bHYGIENE\b",
        r"\bSIDA\b",
    ],
    
    # 15. CULTURE & LOISIRS
    "CULTURE_LOISIRS": [
        r"\bCULTURE\b",
        r"\bARTS\b",
        r"\bSPORTS\b",
        r"\bJEUNESSE\b",
        r"\bLOISIRS\b",
        r"\bTOURISME\b",
        r"\bHOTELLERIE\b",
    ],
    
    # 16. ÉDUCATION NATIONALE
    "EDUCATION_NATIONALE": [
        r"\bEDUCATION\b",
        r"\bENSEIGNEMENT\b",
        r"\bUNIVERSIT\b",
        r"\bEPST\b",
        r"\bESU\b",
        r"\bRECHERCHE\s+SCIENTIFIQUE\b",
        r"\bALPHABETISATION\b",
    ],
    
    # 17. AFFAIRES SOCIALES
    "AFFAIRES_SOCIALES": [
        r"\bAFFAIRES\s+SOCIALES\b",
        r"\bGENRE\b",
        r"\bFAMILLE\b",
        r"\bFEMININE\b",
        r"\bSOLIDARITE\b",
        r"\bACTION\s+HUMANITAIRE\b",
        r"\bTRAVAIL\b",
        r"\bPREVOYANCE\s+SOCIALES\b",
        r"\bFONCTION\s+PUBLIQUE\b",
        r"\bEMPLOI\b",
    ],
    
    # 18. AGRICULTURE
    "AGRICULTURE_PECHE": [
        r"\bAGRICULTURE\b",
        r"\bPECHE\b",
        r"\bELEVAGE\b",
        r"\bDEVELOPPEMENT\s+RURAL\b",
    ],
    
    # 19. MINES & ÉNERGIE
    "MINES_ENERGIE": [
        r"\bMINES\b",
        r"\bHYDROCARBURE\b",
        r"\bPETROLE\b",
        r"\bENERGIE\b",
        r"\bELECTRIC\b",
        r"\bRESSOURCES\s+HYDRAULIQUES\b",
    ],
    
    # 20. TRANSPORTS
    "TRANSPORTS": [
        r"\bTRANSPORT\b",
        r"\bCOMMUNICATIONS\b",
        r"\bVOIES\s+DE\s+COMMUNICATION\b",
        r"\bROUTE\b",
        r"\bVOIRIE\b",
    ],
    
    # 21. COMMUNICATION & MÉDIAS
    "COMMUNICATION_MEDIAS": [
        r"\bINFORMATION\b",
        r"\bPRESSE\b",
        r"\bMEDIAS\b",
        r"\bPOSTES\b",
        r"\bTELEGRAPHES\b",
        r"\bTELECOMMUNICATIONS\b",
    ],
    
    # 22. AFFAIRES ÉTRANGÈRES
    "AFFAIRES_ETRANGERES": [
        r"\bAFFAIRES\s+ETRANGER\b",
        r"\bRELATIONS\s+EXTERIEURES\b",
        r"\bCOOPERATION\s+INTERNATIONALE\b",
        r"\bDIPLOMATIE\b",
    ],
    
    # 23. AFFAIRES FONCIÈRES
    "AFFAIRES_FONCIERES": [
        r"\bAFFAIRES\s+FONCIERES\b",
        r"\bCADASTRE\b",
        r"\bTITRES\s+FONCIERS\b",
    ],
    
    # 24. ADMINISTRATION DU TERRITOIRE
    "ADMINISTRATION_TERRITOIRE": [
        r"\bADMINISTRATION\s+DU\s+TERRITOIRE\b",
        r"\bTERRITOIRE\b",
        r"\bDECENTRALISATION\b",
        r"\bVILLES\s+ET\s+PROVINCES\b",
    ],
}

def attribution_fonction_definitive(ministere: str) -> str:
    """Attribution définitive selon le mapping validé"""
    if not ministere or pd.isna(ministere):
        return "NON_CLASSIFIE"
    
    n = normalize_basic(ministere)
    
    if n == "":
        return "NON_CLASSIFIE"
    
    for fonction, patterns in MAPPING_FONCTIONS_DEFINITIF.items():
        for pattern in patterns:
            if re.search(pattern, n, flags=re.IGNORECASE):
                return fonction
    
    return "NON_CLASSIFIE"

# Test final
print("\n🧪 TEST DU MAPPING DÉFINITIF")
print("="*60)

all_ministeres['Fonction_Definitive'] = all_ministeres['Ministere'].apply(attribution_fonction_definitive)

# Statistiques
final_stats = all_ministeres.groupby('Fonction_Definitive').size().sort_values(ascending=False)

total = len(all_ministeres)
print(f"\n📊 DISTRIBUTION PAR FONCTION ({len(MAPPING_FONCTIONS_DEFINITIF)} fonctions)")
print("="*60)

for fonction, count in final_stats.items():
    pct = (count / total) * 100
    if fonction == "NON_CLASSIFIE":
        status = "❌" if pct > 5 else "⚠️" if pct > 2 else "✅"
    else:
        status = "✅"
    print(f"{status} {fonction:30s} : {count:3d} ({pct:5.1f}%)")

# NON_CLASSIFIÉS restants
non_class_final = all_ministeres[all_ministeres['Fonction_Definitive'] == 'NON_CLASSIFIE']
if len(non_class_final) > 0:
    print(f"\n⚠️ RESTENT NON_CLASSIFIÉS ({len(non_class_final)}) :")
    print("-" * 60)
    for idx, row in non_class_final.iterrows():
        print(f"   • {row['Ministere']}")
else:
    print(f"\n🎉 100% DES MINISTÈRES SONT CLASSIFIÉS !")

# Détail par fonction
print("\n" + "="*60)
print("📋 DÉTAIL DU REGROUPEMENT FINAL")
print("="*60)

for fonction in sorted(MAPPING_FONCTIONS_DEFINITIF.keys()):
    ministeres = all_ministeres[all_ministeres['Fonction_Definitive'] == fonction]['Ministere'].tolist()
    if ministeres:
        print(f"\n🏛️ {fonction} ({len(ministeres)} ministères)")
        print("-" * 60)
        for i, m in enumerate(ministeres, 1):
            print(f"   {i}. {m}")

print("\n" + "="*60)
print(f"✅ MAPPING FINALISÉ : {len(MAPPING_FONCTIONS_DEFINITIF)} fonctions")
print(f"✅ Taux de classification : {((total - len(non_class_final)) / total * 100):.1f}%")
print("="*60)

🔧 MAPPING FINAL DÉFINITIF

🧪 TEST DU MAPPING DÉFINITIF

📊 DISTRIBUTION PAR FONCTION (24 fonctions)
✅ SERVICES_GENERAUX              :  15 ( 20.8%)
✅ ECONOMIE                       :   6 (  8.3%)
✅ DEPENSES_EXCEPTIONNELLES       :   5 (  6.9%)
✅ AFFAIRES_SOCIALES              :   4 (  5.6%)
✅ JUSTICE                        :   4 (  5.6%)
✅ MINES_ENERGIE                  :   3 (  4.2%)
✅ ADMINISTRATION_TERRITOIRE      :   3 (  4.2%)
✅ CULTURE_LOISIRS                :   3 (  4.2%)
✅ PORTEFEUILLE_PME               :   3 (  4.2%)
✅ PARLEMENT                      :   3 (  4.2%)
✅ TRAVAUX_PUBLICS                :   2 (  2.8%)
✅ DEFENSE_SECURITE               :   2 (  2.8%)
✅ COMMUNICATION_MEDIAS           :   2 (  2.8%)
✅ AGRICULTURE_PECHE              :   2 (  2.8%)
✅ PRESIDENCE                     :   2 (  2.8%)
✅ EDUCATION_NATIONALE            :   2 (  2.8%)
✅ FINANCES_PUBLIQUES             :   2 (  2.8%)
✅ PRIMATURE                      :   2 (  2.8%)
✅ AFFAIRES_ETRANGERES            :   

In [16]:
# ============================================================
# GÉNÉRATION COMPLÈTE DES 5 FICHIERS AVEC NOUVELLES FONCTIONS
# ============================================================

print("🚀 GÉNÉRATION COMPLÈTE - 5 FICHIERS")
print("="*60)

# Définir les noms de fichiers
OUT_2005_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2005_OK.xlsx"
OUT_2014_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_2005_2014_OK.xlsx"
OUT_2023_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_2014_2023_OK.xlsx"
OUT_WIDE_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2023_WIDE_OK.xlsx"
OUT_LONG_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2023_LONG_OK.xlsx"

# Fonction d'enrichissement
def enrich_avec_nouvelles_fonctions(df, col="Ministere"):
    df = df.copy()
    df["Ministere_raw"] = df[col].astype(str)
    df["Ministere_canonique"] = df["Ministere_raw"].apply(canonicalize_ministere_short)
    df["Fonction"] = df["Ministere_canonique"].apply(attribution_fonction_definitive)
    df["Ministere_key"] = df["Ministere_canonique"].apply(normalize_basic)
    df[col] = df["Ministere_canonique"]
    return df

# ============================================================
# ÉTAPE 1 : EXTRACTION DES 3 RAPPORTS
# ============================================================

print("\n" + "="*60)
print("ÉTAPE 1/2 : EXTRACTION DES 3 RAPPORTS BCC")
print("="*60)

# --- RAPPORT 2005 ---
print("\n📄 1/3 Rapport 2005 (1996-2005)...")
df2005 = extract_2005(PDF_2005, PAGE_2005)
df2005 = enrich_avec_nouvelles_fonctions(df2005, col="Ministere")
print(f"   ✅ {df2005.shape[0]} ministères | {df2005['Fonction'].nunique()} fonctions")

cols_2005 = ["Ministere", "Fonction"] + [str(y) for y in range(1996, 2006)]
df2005[cols_2005].to_excel(OUT_2005_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_2005_OK.name}")

# --- RAPPORT 2014 ---
print("\n📄 2/3 Rapport 2014 (2005-2014)...")
import camelot

tables = camelot.read_pdf(
    str(PDF_2014),
    pages=str(PAGE_2014),
    flavor="stream",
    strip_text="\n",
    split_text=True
)

best_table = max(tables, key=lambda t: t.df.shape[1])
df_raw = best_table.df

df2014 = df_raw.copy()
df2014 = df2014.dropna(axis=0, how='all').reset_index(drop=True)

years_2014 = [str(y) for y in range(2005, 2015)]
df2014 = df2014.iloc[:, :11].copy()
df2014.columns = ["Ministere"] + years_2014

df2014["Ministere"] = df2014["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df2014 = df2014[df2014["Ministere"].notna()].copy()
df2014 = df2014[~df2014["Ministere"].str.lower().isin(['nan', 'none', '', 'total'])].copy()
df2014 = df2014[~df2014["Ministere"].str.lower().str.startswith('source')].copy()
df2014 = df2014.reset_index(drop=True)

for year in years_2014:
    df2014[year] = df2014[year].apply(to_number)

df2014 = enrich_avec_nouvelles_fonctions(df2014, col="Ministere")
print(f"   ✅ {df2014.shape[0]} ministères | {df2014['Fonction'].nunique()} fonctions")

cols_2014 = ["Ministere", "Fonction"] + years_2014
df2014[cols_2014].to_excel(OUT_2014_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_2014_OK.name}")

# --- RAPPORT 2023 ---
print("\n📄 3/3 Rapport 2023 (2014-2023)...")
df2023 = extract_2023(PDF_2023)
df2023 = enrich_avec_nouvelles_fonctions(df2023, col="Ministere")
print(f"   ✅ {df2023.shape[0]} ministères | {df2023['Fonction'].nunique()} fonctions")

cols_2023 = ["Ministere", "Fonction"] + [str(y) for y in range(2014, 2024)]
df2023[cols_2023].to_excel(OUT_2023_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_2023_OK.name}")

# ============================================================
# ÉTAPE 2 : CONSOLIDATION WIDE + LONG
# ============================================================

print("\n" + "="*60)
print("ÉTAPE 2/2 : CONSOLIDATION WIDE + LONG")
print("="*60)

# Fusion des 3 périodes
print("\n🔗 Fusion des 3 périodes...")
m1 = merge_two(df2005, df2014, prefer_right_years=["2005"])
m2 = merge_two(m1, df2023, prefer_right_years=["2014"])
print(f"   ✅ Fusion complète : {m2.shape}")

# Format WIDE
print("\n📊 Génération format WIDE...")
all_years = sorted([c for c in m2.columns if re.match(r"^(19|20)\d{2}$", str(c))], key=lambda x: int(x))
wide_cols = ["Ministere", "Fonction"] + all_years
wide = m2[wide_cols].copy()
wide = wide[wide["Ministere"].notna() & (wide["Ministere"].astype(str).str.strip() != "")].reset_index(drop=True)

print(f"   Période  : {all_years[0]} → {all_years[-1]} ({len(all_years)} années)")
print(f"   Données  : {wide.shape[0]} ministères × {wide.shape[1]} colonnes")

wide.to_excel(OUT_WIDE_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_WIDE_OK.name}")

# Format LONG
print("\n📊 Génération format LONG...")
long_df = wide.melt(
    id_vars=["Ministere", "Fonction"],
    var_name="Annee",
    value_name="Budget_Depense_Courante"
)
long_df["Annee"] = long_df["Annee"].astype(int)

print(f"   Observations : {len(long_df):,}")
print(f"   Ministères   : {long_df['Ministere'].nunique()}")
print(f"   Fonctions    : {long_df['Fonction'].nunique()}")
print(f"   Période      : {long_df['Annee'].min()} → {long_df['Annee'].max()}")

long_df.to_excel(OUT_LONG_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_LONG_OK.name}")

# ============================================================
# RÉSUMÉ FINAL
# ============================================================

print("\n" + "="*60)
print("✅ GÉNÉRATION TERMINÉE - 5 FICHIERS CRÉÉS")
print("="*60)

fichiers_generes = [OUT_2005_OK, OUT_2014_OK, OUT_2023_OK, OUT_WIDE_OK, OUT_LONG_OK]

print(f"\n📂 Dossier : {OUTPUT_DIR.absolute()}\n")

for fichier in fichiers_generes:
    if fichier.exists():
        taille = fichier.stat().st_size / 1024
        print(f"✅ {fichier.name:50s} ({taille:6.1f} KB)")
    else:
        print(f"❌ {fichier.name:50s} (MANQUANT)")

# Statistiques finales
print("\n" + "="*60)
print("📊 STATISTIQUES FINALES")
print("="*60)

print(f"\nPériode couverte     : {long_df['Annee'].min()} → {long_df['Annee'].max()}")
print(f"Nombre d'années      : {long_df['Annee'].nunique()}")
print(f"Nombre de ministères : {long_df['Ministere'].nunique()}")
print(f"Nombre de fonctions  : {long_df['Fonction'].nunique()}")
print(f"Observations totales : {len(long_df):,}")
print(f"Valeurs renseignées  : {long_df['Budget_Depense_Courante'].notna().sum():,}")
print(f"Valeurs manquantes   : {long_df['Budget_Depense_Courante'].isna().sum():,} ({long_df['Budget_Depense_Courante'].isna().mean()*100:.1f}%)")

# Distribution par fonction
print(f"\n📋 Distribution par fonction (Top 10) :")
print("-" * 60)

for fonction, count in long_df['Fonction'].value_counts().head(10).items():
    pct = (count / len(long_df)) * 100
    print(f"   {fonction:30s} : {count:6,} obs ({pct:5.1f}%)")

# Aperçu du fichier LONG
print(f"\n📋 Aperçu du fichier LONG (10 premières lignes) :")
print("-" * 60)
print(long_df.head(10).to_string(index=False))

print("\n" + "="*60)
print("🎉 PRÊT POUR LE MACHINE LEARNING !")
print("="*60)

🚀 GÉNÉRATION COMPLÈTE - 5 FICHIERS

ÉTAPE 1/2 : EXTRACTION DES 3 RAPPORTS BCC

📄 1/3 Rapport 2005 (1996-2005)...


C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\21138

   ✅ 64 ministères | 25 fonctions
   💾 Budget_Depenses_Courantes_1996_2005_OK.xlsx

📄 2/3 Rapport 2014 (2005-2014)...
   ✅ 60 ministères | 25 fonctions
   💾 Budget_Depenses_Courantes_2005_2014_OK.xlsx

📄 3/3 Rapport 2023 (2014-2023)...


C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())


   ✅ 58 ministères | 25 fonctions
   💾 Budget_Depenses_Courantes_2014_2023_OK.xlsx

ÉTAPE 2/2 : CONSOLIDATION WIDE + LONG

🔗 Fusion des 3 périodes...
   ✅ Fusion complète : (72, 31)

📊 Génération format WIDE...
   Période  : 1996 → 2023 (28 années)
   Données  : 72 ministères × 30 colonnes
   💾 Budget_Depenses_Courantes_1996_2023_WIDE_OK.xlsx

📊 Génération format LONG...
   Observations : 2,016
   Ministères   : 72
   Fonctions    : 25
   Période      : 1996 → 2023
   💾 Budget_Depenses_Courantes_1996_2023_LONG_OK.xlsx

✅ GÉNÉRATION TERMINÉE - 5 FICHIERS CRÉÉS

📂 Dossier : c:\Users\LEGION\Music\Memoire Master\scripts\..\data\processed

✅ Budget_Depenses_Courantes_1996_2005_OK.xlsx        (  10.2 KB)
✅ Budget_Depenses_Courantes_2005_2014_OK.xlsx        (   9.9 KB)
✅ Budget_Depenses_Courantes_2014_2023_OK.xlsx        (  10.0 KB)
✅ Budget_Depenses_Courantes_1996_2023_WIDE_OK.xlsx   (  17.8 KB)
✅ Budget_Depenses_Courantes_1996_2023_LONG_OK.xlsx   (  49.6 KB)

📊 STATISTIQUES FINALES

Période

In [17]:
# ============================================================
# GÉNÉRATION COMPLÈTE - 5 FICHIERS (SANS TOTAUX)
# ============================================================

print("🚀 GÉNÉRATION COMPLÈTE - 5 FICHIERS (VERSION CORRIGÉE)")
print("="*60)

# Noms de fichiers
OUT_2005_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2005_OK.xlsx"
OUT_2014_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_2005_2014_OK.xlsx"
OUT_2023_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_2014_2023_OK.xlsx"
OUT_WIDE_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2023_WIDE_OK.xlsx"
OUT_LONG_OK = OUTPUT_DIR / "Budget_Depenses_Courantes_1996_2023_LONG_OK.xlsx"

# ============================================================
# FONCTION DE FILTRAGE DES TOTAUX/SOUS-TOTAUX
# ============================================================

def supprimer_lignes_totaux(df, col="Ministere"):
    """
    Supprime les lignes de totaux et sous-totaux :
    - I.Institutions politiques
    - II.Ministères
    - III.Autres services
    - Total
    - Lignes vides
    """
    # Patterns à supprimer
    patterns_totaux = [
        r"^I\..*INSTITUTIONS.*POLITIQUES",
        r"^II\..*MINISTERES",
        r"^III\..*AUTRES.*SERVICES",
        r"^TOTAL$",
        r"^I\.INSTITUTIONS\s+POLITIQUES$",
        r"^II\.MINISTERES$",
        r"^III\.AUTRES\s+SERVICES$",
    ]
    
    keep_indices = []
    
    for i, row in df.iterrows():
        ministere = str(row[col]).strip()
        
        # Vérifier si vide ou NaN
        if ministere == "" or ministere.lower() in ["nan", "none"]:
            continue
        
        # Vérifier si c'est une ligne de total
        is_total = False
        ministere_norm = normalize_basic(ministere)
        
        for pattern in patterns_totaux:
            if re.match(pattern, ministere_norm, flags=re.IGNORECASE):
                is_total = True
                break
        
        # Garder seulement si ce n'est pas un total
        if not is_total:
            keep_indices.append(i)
    
    return df.loc[keep_indices].reset_index(drop=True)

# Fonction d'enrichissement
def enrich_avec_nouvelles_fonctions(df, col="Ministere"):
    df = df.copy()
    
    # SUPPRIMER LES TOTAUX D'ABORD
    df = supprimer_lignes_totaux(df, col=col)
    
    # Puis enrichir
    df["Ministere_raw"] = df[col].astype(str)
    df["Ministere_canonique"] = df["Ministere_raw"].apply(canonicalize_ministere_short)
    df["Fonction"] = df["Ministere_canonique"].apply(attribution_fonction_definitive)
    df["Ministere_key"] = df["Ministere_canonique"].apply(normalize_basic)
    df[col] = df["Ministere_canonique"]
    return df

# ============================================================
# ÉTAPE 1 : EXTRACTION DES 3 RAPPORTS
# ============================================================

print("\n" + "="*60)
print("ÉTAPE 1/2 : EXTRACTION DES 3 RAPPORTS (SANS TOTAUX)")
print("="*60)

# --- RAPPORT 2005 ---
print("\n📄 1/3 Rapport 2005 (1996-2005)...")
df2005 = extract_2005(PDF_2005, PAGE_2005)
df2005 = enrich_avec_nouvelles_fonctions(df2005, col="Ministere")
print(f"   ✅ {df2005.shape[0]} ministères (après suppression totaux)")
print(f"   📊 {df2005['Fonction'].nunique()} fonctions")

cols_2005 = ["Ministere", "Fonction"] + [str(y) for y in range(1996, 2006)]
df2005[cols_2005].to_excel(OUT_2005_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_2005_OK.name}")

# --- RAPPORT 2014 ---
print("\n📄 2/3 Rapport 2014 (2005-2014)...")
import camelot

tables = camelot.read_pdf(
    str(PDF_2014),
    pages=str(PAGE_2014),
    flavor="stream",
    strip_text="\n",
    split_text=True
)

best_table = max(tables, key=lambda t: t.df.shape[1])
df_raw = best_table.df

df2014 = df_raw.copy()
df2014 = df2014.dropna(axis=0, how='all').reset_index(drop=True)

years_2014 = [str(y) for y in range(2005, 2015)]
df2014 = df2014.iloc[:, :11].copy()
df2014.columns = ["Ministere"] + years_2014

df2014["Ministere"] = df2014["Ministere"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df2014 = df2014[df2014["Ministere"].notna()].copy()
df2014 = df2014[~df2014["Ministere"].str.lower().isin(['nan', 'none', '', 'total'])].copy()
df2014 = df2014[~df2014["Ministere"].str.lower().str.startswith('source')].copy()
df2014 = df2014.reset_index(drop=True)

for year in years_2014:
    df2014[year] = df2014[year].apply(to_number)

df2014 = enrich_avec_nouvelles_fonctions(df2014, col="Ministere")
print(f"   ✅ {df2014.shape[0]} ministères (après suppression totaux)")
print(f"   📊 {df2014['Fonction'].nunique()} fonctions")

cols_2014 = ["Ministere", "Fonction"] + years_2014
df2014[cols_2014].to_excel(OUT_2014_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_2014_OK.name}")

# --- RAPPORT 2023 ---
print("\n📄 3/3 Rapport 2023 (2014-2023)...")
df2023 = extract_2023(PDF_2023)
df2023 = enrich_avec_nouvelles_fonctions(df2023, col="Ministere")
print(f"   ✅ {df2023.shape[0]} ministères (après suppression totaux)")
print(f"   📊 {df2023['Fonction'].nunique()} fonctions")

cols_2023 = ["Ministere", "Fonction"] + [str(y) for y in range(2014, 2024)]
df2023[cols_2023].to_excel(OUT_2023_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_2023_OK.name}")

# ============================================================
# ÉTAPE 2 : CONSOLIDATION WIDE + LONG
# ============================================================

print("\n" + "="*60)
print("ÉTAPE 2/2 : CONSOLIDATION WIDE + LONG")
print("="*60)

# Fusion
print("\n🔗 Fusion des 3 périodes...")
m1 = merge_two(df2005, df2014, prefer_right_years=["2005"])
m2 = merge_two(m1, df2023, prefer_right_years=["2014"])
print(f"   ✅ Fusion : {m2.shape}")

# Format WIDE
print("\n📊 Génération WIDE...")
all_years = sorted([c for c in m2.columns if re.match(r"^(19|20)\d{2}$", str(c))], key=lambda x: int(x))
wide_cols = ["Ministere", "Fonction"] + all_years
wide = m2[wide_cols].copy()
wide = wide[wide["Ministere"].notna() & (wide["Ministere"].astype(str).str.strip() != "")].reset_index(drop=True)

# Supprimer les totaux dans WIDE aussi (au cas où)
wide = supprimer_lignes_totaux(wide, col="Ministere")

print(f"   Période : {all_years[0]} → {all_years[-1]} ({len(all_years)} années)")
print(f"   Données : {wide.shape[0]} ministères × {wide.shape[1]} colonnes")

wide.to_excel(OUT_WIDE_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_WIDE_OK.name}")

# Format LONG
print("\n📊 Génération LONG...")
long_df = wide.melt(
    id_vars=["Ministere", "Fonction"],
    var_name="Annee",
    value_name="Budget_Depense_Courante"
)
long_df["Annee"] = long_df["Annee"].astype(int)

print(f"   Observations : {len(long_df):,}")
print(f"   Ministères   : {long_df['Ministere'].nunique()}")
print(f"   Fonctions    : {long_df['Fonction'].nunique()}")

long_df.to_excel(OUT_LONG_OK, index=False, engine='openpyxl')
print(f"   💾 {OUT_LONG_OK.name}")

# ============================================================
# RÉSUMÉ FINAL
# ============================================================

print("\n" + "="*60)
print("✅ GÉNÉRATION TERMINÉE - 5 FICHIERS (SANS TOTAUX)")
print("="*60)

fichiers = [OUT_2005_OK, OUT_2014_OK, OUT_2023_OK, OUT_WIDE_OK, OUT_LONG_OK]

print(f"\n📂 {OUTPUT_DIR.absolute()}\n")

for f in fichiers:
    if f.exists():
        taille = f.stat().st_size / 1024
        print(f"✅ {f.name:50s} ({taille:6.1f} KB)")

print("\n" + "="*60)
print("📊 STATISTIQUES FINALES")
print("="*60)

print(f"\nPériode couverte     : {long_df['Annee'].min()} → {long_df['Annee'].max()}")
print(f"Années               : {long_df['Annee'].nunique()}")
print(f"Ministères           : {long_df['Ministere'].nunique()}")
print(f"Fonctions            : {long_df['Fonction'].nunique()}")
print(f"Observations         : {len(long_df):,}")
print(f"Valeurs renseignées  : {long_df['Budget_Depense_Courante'].notna().sum():,} ({long_df['Budget_Depense_Courante'].notna().mean()*100:.1f}%)")
print(f"Valeurs manquantes   : {long_df['Budget_Depense_Courante'].isna().sum():,} ({long_df['Budget_Depense_Courante'].isna().mean()*100:.1f}%)")

print(f"\n📋 Top 10 fonctions par nombre d'observations :")
print("-" * 60)
for fonction, count in long_df['Fonction'].value_counts().head(10).items():
    pct = (count / len(long_df)) * 100
    print(f"   {fonction:30s} : {count:6,} ({pct:5.1f}%)")

print(f"\n📋 Aperçu LONG (15 premières lignes) :")
print("-" * 60)
print(long_df.head(15).to_string(index=False))

print("\n" + "="*60)
print("🎉 DONNÉES PRÊTES POUR LE MACHINE LEARNING !")
print("="*60)

🚀 GÉNÉRATION COMPLÈTE - 5 FICHIERS (VERSION CORRIGÉE)

ÉTAPE 1/2 : EXTRACTION DES 3 RAPPORTS (SANS TOTAUX)

📄 1/3 Rapport 2005 (1996-2005)...


C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())


   ✅ 61 ministères (après suppression totaux)
   📊 25 fonctions


C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())
C:\Users\LEGION\AppData\Local\Temp\ipykernel_17180\2113828521.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: "" if x is None else str(x).replace("\n", " ").strip())


PermissionError: [Errno 13] Permission denied: '..\\data\\processed\\Budget_Depenses_Courantes_1996_2005_OK.xlsx'